# SPX VIX-Style Term Structure — v0.7 Production Notebook

**Methodology:** `v0.7_exchange_calendar_fred_sofr_1600`

This notebook builds VIX-style implied-volatility term structures for custom target tenors using SPX/SPXW option chains from ThetaData v3.

Key methodology choices:

- SPX/SPXW option quotes at **16:00 ET**
- **FRED SOFR** as the risk-free rate source
- **Exchange calendar / XNYS** as the trading-calendar source
- Friday-cycle expiration universe:
  - normal Friday expirations
  - holiday-adjusted expirations shifted to the prior trading day when Friday is closed
- Linear interpolation in **total variance**
- Local raw-chain cache
- Processed long-format history with idempotent upsert

**Path behavior:** this notebook stores data at the project root, e.g. `vrp_project/data/...`, not inside `vrp_project/notebooks/data/...`.

**Data-source architecture:**

- Trading calendar: `pandas_market_calendars` / `XNYS`
- Rates: FRED SOFR CSV
- Options: ThetaData SPX/SPXW option chains


In [1]:
# ============================================================
# Imports and global settings
# ============================================================

import io
import math
import requests
import numpy as np
import pandas as pd
import pandas_market_calendars as mcal

from pathlib import Path
from datetime import date, datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed


# ThetaData v3 local terminal endpoint
BASE_URL_V3 = "http://127.0.0.1:25503/v3"


# VIX calculation time: 4:00 PM ET
# Milliseconds after midnight
CALC_TIME_MS = 16 * 60 * 60 * 1000


# VIX methodology constants
MINUTES_PER_DAY = 1440
MINUTES_PER_YEAR = 525600
MINUTES_30_DAYS = 43200


# Rate settings
DEFAULT_RATE_SYMBOL = "SOFR"
DEFAULT_RISK_FREE_RATE = 0.05


# Target tenors for the first custom term-structure build
TARGET_TENOR_DAYS = [9, 12, 15, 18, 21, 24, 27, 30, 33]
PRIMARY_VALIDATION_TENOR_DAYS = 30


# Methodology labels
METHODOLOGY_VERSION = "v0.7_exchange_calendar_fred_sofr_1600"
EXPIRATION_UNIVERSE = "friday_cycle_holiday_adjusted"
TRADING_CALENDAR_SOURCE = "exchange_calendar_XNYS"
RATE_SOURCE = "FRED_SOFR"
OPTION_SOURCE = "THETADATA_SPX_SPXW"


In [2]:
# ============================================================
# Date and time helpers
# ============================================================

def yyyymmdd_to_date(x):
    """
    Convert 20240116 to datetime.date(2024, 1, 16).
    """
    s = str(int(x))
    return date(int(s[:4]), int(s[4:6]), int(s[6:8]))


def yyyymmdd_to_dash_string(x):
    """
    Convert 20240116 to '2024-01-16'.
    """
    return yyyymmdd_to_date(x).strftime("%Y-%m-%d")


def int_date_from_date(dt):
    """
    Convert datetime.date to YYYYMMDD integer.
    """
    return int(dt.strftime("%Y%m%d"))


def ms_to_time_string(ms_of_day):
    """
    Convert milliseconds after midnight to ThetaData v3 time string.

    Example:
        57600000 -> '16:00:00.000'
    """
    total_seconds = int(ms_of_day // 1000)

    hours = total_seconds // 3600
    minutes = (total_seconds % 3600) // 60
    seconds = total_seconds % 60

    return f"{hours:02d}:{minutes:02d}:{seconds:02d}.000"


def cache_time_label(calc_time_ms):
    """
    Convert calc time to compact label.

    Example:
        57600000 -> '160000'
    """
    time_str = ms_to_time_string(calc_time_ms)
    return time_str.replace(":", "").replace(".000", "")


def is_third_friday(dt):
    """
    True if date is the standard monthly SPX expiration Friday.
    """
    return dt.weekday() == 4 and 15 <= dt.day <= 21


QUOTE_TIME_LABEL = cache_time_label(CALC_TIME_MS)


In [3]:
# ============================================================
# Project paths
# ============================================================

# This notebook usually runs from:
#     <project_root>/notebooks
#
# All data should be saved outside the notebooks folder:
#     <project_root>/data

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name.lower() == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
EXTERNAL_DATA_DIR = DATA_DIR / "external"

CHAIN_CACHE_DIR = RAW_DATA_DIR / "thetadata_chains"
TERM_STRUCTURE_HISTORY_CSV = PROCESSED_DATA_DIR / "vix_term_structure_history.csv"
TERM_STRUCTURE_HISTORY_PARQUET = PROCESSED_DATA_DIR / "vix_term_structure_history.parquet"
SPX_TRADING_DATES_CSV = PROCESSED_DATA_DIR / "spx_trading_dates.csv"

RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
EXTERNAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
CHAIN_CACHE_DIR.mkdir(parents=True, exist_ok=True)

print("Current directory:", CURRENT_DIR)
print("Project root:", PROJECT_ROOT)
print("Data directory:", DATA_DIR)
print("Processed history CSV:", TERM_STRUCTURE_HISTORY_CSV)
print("External data directory:", EXTERNAL_DATA_DIR)
print("SPX trading calendar CSV:", SPX_TRADING_DATES_CSV)
print("Raw chain cache:", CHAIN_CACHE_DIR)


Current directory: C:\Users\patri\vrp_project\notebooks
Project root: C:\Users\patri\vrp_project
Data directory: C:\Users\patri\vrp_project\data
Processed history CSV: C:\Users\patri\vrp_project\data\processed\vix_term_structure_history.csv
External data directory: C:\Users\patri\vrp_project\data\external
SPX trading calendar CSV: C:\Users\patri\vrp_project\data\processed\spx_trading_dates.csv
Raw chain cache: C:\Users\patri\vrp_project\data\raw\thetadata_chains


## Data loaders

The v0.7 production setup uses ThetaData for SPX/SPXW options only. Trading dates come from an exchange calendar, and SOFR comes from FRED.


In [4]:
# ============================================================
# ThetaData v3 expiration loader
# ============================================================

def list_expirations_v3(symbol, verbose=False):
    """
    Get option expirations from ThetaData v3.

    v3 endpoint returns CSV text like:
        symbol,expiration
        "SPX","2024-02-16"

    This function converts expiration dates to YYYYMMDD integers.
    """
    url = BASE_URL_V3 + "/option/list/expirations"
    params = {"symbol": symbol}

    r = requests.get(url, params=params, timeout=60)

    if verbose:
        print("URL:", r.url)
        print("Status code:", r.status_code)
        print(r.text[:1000])

    if r.status_code != 200:
        raise RuntimeError(
            f"ThetaData expiration request failed.\n"
            f"URL: {r.url}\n"
            f"Status code: {r.status_code}\n"
            f"Response text:\n{r.text[:2000]}"
        )

    df = pd.read_csv(io.StringIO(r.text))

    if df.empty:
        raise ValueError(f"No expirations returned for {symbol}")

    if "expiration" not in df.columns:
        raise ValueError(f"Could not find 'expiration' column. Columns are: {list(df.columns)}")

    expirations = (
        pd.to_datetime(df["expiration"])
        .dt.strftime("%Y%m%d")
        .astype(int)
        .tolist()
    )

    return sorted(expirations)


spx_exps = list_expirations_v3("SPX")
spxw_exps = list_expirations_v3("SPXW")

print("SPX expirations loaded:", len(spx_exps))
print("SPXW expirations loaded:", len(spxw_exps))
print("SPX sample:", spx_exps[:3], "...", spx_exps[-3:])
print("SPXW sample:", spxw_exps[:3], "...", spxw_exps[-3:])


SPX expirations loaded: 203
SPXW expirations loaded: 2203
SPX sample: [20120616, 20120721, 20120818] ... [20891219, 20900116, 20900618]
SPXW sample: [20120601, 20120608, 20120622] ... [20881018, 20890331, 20890630]


In [5]:
# ============================================================
# FRED SOFR loader
# ============================================================

FRED_SOFR_CSV = EXTERNAL_DATA_DIR / "fred_sofr_history.csv"
LEGACY_FRED_SOFR_CSV = PROCESSED_DATA_DIR / "fred_sofr_history.csv"
FRED_SOFR_URL = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=SOFR"

_FRED_SOFR_CACHE = None


def load_fred_sofr_history(force_refresh=False, verbose=False):
    """
    Load SOFR history from FRED and cache it in memory.

    SOFR is quoted in percent form in the source file:
        5.32 means 5.32%

    This function converts it to decimal form:
        0.0532
    """
    global _FRED_SOFR_CACHE

    if _FRED_SOFR_CACHE is not None and not force_refresh:
        return _FRED_SOFR_CACHE.copy()

    if FRED_SOFR_CSV.exists() and not force_refresh:
        raw_df = pd.read_csv(FRED_SOFR_CSV)
    elif LEGACY_FRED_SOFR_CSV.exists() and not force_refresh:
        raw_df = pd.read_csv(LEGACY_FRED_SOFR_CSV)
        raw_df.to_csv(FRED_SOFR_CSV, index=False)
    else:
        raw_df = pd.read_csv(FRED_SOFR_URL)
        raw_df.to_csv(FRED_SOFR_CSV, index=False)

    sofr_df = raw_df.copy()
    sofr_df["observation_date"] = pd.to_datetime(sofr_df["observation_date"])
    sofr_df["trade_date"] = sofr_df["observation_date"].dt.strftime("%Y%m%d").astype(int)
    sofr_df["rate_pct"] = pd.to_numeric(sofr_df["SOFR"], errors="coerce")
    sofr_df = sofr_df.dropna(subset=["rate_pct"]).copy()
    sofr_df["rate_decimal"] = sofr_df["rate_pct"] / 100.0

    sofr_df = sofr_df[
        ["trade_date", "observation_date", "rate_pct", "rate_decimal"]
    ].sort_values("trade_date").reset_index(drop=True)

    _FRED_SOFR_CACHE = sofr_df.copy()

    if verbose:
        print("SOFR rows:", len(sofr_df))
        print("First SOFR date:", sofr_df["trade_date"].min())
        print("Last SOFR date:", sofr_df["trade_date"].max())
        print("Saved to:", FRED_SOFR_CSV)

    return sofr_df.copy()


def update_fred_sofr_history(force_refresh=False):
    """
    Refresh/load local FRED SOFR history and print a summary.
    """
    return load_fred_sofr_history(force_refresh=force_refresh, verbose=True)


def get_interest_rate_for_date_v3(symbol, trade_date, verbose=False):
    """
    Return SOFR for a trade date using local FRED history.

    Uses the latest available SOFR observation on or before trade_date.
    This avoids ThetaData interest-rate subscription limits before 2024.

    The function name is kept as get_interest_rate_for_date_v3 so the VIX engine
    can call the same interface used in earlier notebook versions.
    """
    symbol_upper = str(symbol).upper()

    if symbol_upper != "SOFR":
        raise ValueError(
            f"This v0.7 notebook currently supports only SOFR. Received: {symbol}"
        )

    sofr_df = load_fred_sofr_history(force_refresh=False, verbose=False)
    trade_date = int(trade_date)

    eligible = sofr_df[sofr_df["trade_date"] <= trade_date].copy()

    if eligible.empty:
        raise RuntimeError(
            f"No SOFR observation available on or before {trade_date}. "
            f"First available SOFR date is {sofr_df['trade_date'].min()}."
        )

    latest = eligible.iloc[-1]

    if verbose:
        print(
            f"Using SOFR {latest['rate_pct']:.4f}% "
            f"from {int(latest['trade_date'])} for trade date {trade_date}"
        )

    return float(latest["rate_decimal"])


In [6]:
# ============================================================
# ThetaData v3 option-chain loader
# ============================================================

def get_chain_at_time(root, expi, trade_date, calc_time_ms, verbose=False):
    """
    Pull all strikes / both calls and puts for one SPX or SPXW expiration
    at a specific time using ThetaData v3.

    Returns dataframe with columns expected by the VIX math:
        root, expiration, strike, right, bid, ask, mid
    """
    request_time = ms_to_time_string(calc_time_ms)
    url = BASE_URL_V3 + "/option/history/quote"

    params = {
        "symbol": root,
        "expiration": yyyymmdd_to_dash_string(expi),
        "strike": "*",
        "right": "both",
        "start_date": yyyymmdd_to_dash_string(trade_date),
        "end_date": yyyymmdd_to_dash_string(trade_date),
        "start_time": request_time,
        "end_time": request_time,
        "interval": "1m",
        "format": "json",
    }

    r = requests.get(url, params=params, timeout=180)

    if verbose:
        print("URL:", r.url)
        print("Status code:", r.status_code)
        print(r.text[:1000])

    if r.status_code != 200:
        raise RuntimeError(
            f"ThetaData option-chain request failed.\n"
            f"URL: {r.url}\n"
            f"Status code: {r.status_code}\n"
            f"Response text:\n{r.text[:2000]}"
        )

    data = r.json()
    df = pd.DataFrame(data)

    if df.empty:
        raise ValueError(
            f"No data returned for {root} {expi} on {trade_date} at {request_time}"
        )

    df["root"] = df["symbol"]
    df["expiration"] = pd.to_datetime(df["expiration"]).dt.strftime("%Y%m%d").astype(int)

    right_map = {
        "CALL": "C",
        "PUT": "P",
        "C": "C",
        "P": "P",
    }

    df["right"] = df["right"].map(right_map)

    if df["right"].isna().any():
        bad_values = df["right"].unique()
        raise ValueError(f"Unknown option right values found: {bad_values}")

    df["bid"] = pd.to_numeric(df["bid"], errors="coerce")
    df["ask"] = pd.to_numeric(df["ask"], errors="coerce")
    df["strike"] = pd.to_numeric(df["strike"], errors="coerce")
    df["mid"] = (df["bid"] + df["ask"]) / 2

    keep_cols = [
        "root",
        "expiration",
        "strike",
        "right",
        "bid",
        "ask",
        "mid",
        "bid_size",
        "ask_size",
        "bid_exchange",
        "ask_exchange",
        "bid_condition",
        "ask_condition",
        "timestamp",
    ]

    return df[keep_cols].copy()


In [7]:
# ============================================================
# SPX trading calendar from exchange calendar
# ============================================================

def yyyymmdd_int_to_iso(date_int):
    """
    Convert YYYYMMDD integer to YYYY-MM-DD string.
    """
    return pd.to_datetime(str(int(date_int)), format="%Y%m%d").strftime("%Y-%m-%d")


def get_spx_trading_dates_from_exchange_calendar(
    start_date,
    end_date,
    calendar_name="XNYS",
):
    """
    Get actual US equity trading sessions from an exchange calendar.

    This replaces ThetaData SPX index EOD as the trading-date source,
    because the user's ThetaData index EOD access starts later than the
    available options-history window.
    """
    cal = mcal.get_calendar(calendar_name)

    start_iso = yyyymmdd_int_to_iso(start_date)
    end_iso = yyyymmdd_int_to_iso(end_date)

    schedule = cal.schedule(
        start_date=start_iso,
        end_date=end_iso,
    )

    trading_dates = (
        pd.Series(schedule.index)
        .dt.strftime("%Y%m%d")
        .astype(int)
        .tolist()
    )

    return trading_dates


def update_spx_trading_dates_file(
    start_date,
    end_date,
    force_refresh=False,
    chunk_days=None,
    calendar_name="XNYS",
):
    """
    Build/update local SPX trading-date file using an exchange calendar.

    Use a calendar end date beyond your backfill end date so expiration selection
    can look ahead to future expirations and holiday Fridays.

    chunk_days is accepted for compatibility with earlier code but is not used.
    """
    if SPX_TRADING_DATES_CSV.exists() and not force_refresh:
        existing = pd.read_csv(SPX_TRADING_DATES_CSV)
    else:
        existing = pd.DataFrame(columns=["trade_date"])

    new_dates = pd.DataFrame({
        "trade_date": get_spx_trading_dates_from_exchange_calendar(
            start_date=start_date,
            end_date=end_date,
            calendar_name=calendar_name,
        )
    })

    combined = (
        pd.concat([existing[["trade_date"]], new_dates], ignore_index=True)
        .drop_duplicates()
        .sort_values("trade_date")
        .reset_index(drop=True)
    )

    combined.to_csv(SPX_TRADING_DATES_CSV, index=False)

    print(f"Saved SPX trading dates to: {SPX_TRADING_DATES_CSV}")
    print("Calendar source:", TRADING_CALENDAR_SOURCE)
    print("Rows:", len(combined))
    print("Min date:", combined["trade_date"].min())
    print("Max date:", combined["trade_date"].max())

    return combined


def load_spx_trading_dates():
    """
    Load locally saved SPX trading dates.
    """
    if not SPX_TRADING_DATES_CSV.exists():
        raise FileNotFoundError(
            f"{SPX_TRADING_DATES_CSV} does not exist yet. "
            "Run update_spx_trading_dates_file first."
        )

    df = pd.read_csv(SPX_TRADING_DATES_CSV)
    return sorted(df["trade_date"].astype(int).unique().tolist())


def get_spx_trade_dates_between(start_date, end_date):
    """
    Return actual SPX trading dates between start_date and end_date
    using the locally saved exchange trading calendar.
    """
    trading_dates = load_spx_trading_dates()

    start_date = int(start_date)
    end_date = int(end_date)

    return [
        int(d) for d in trading_dates
        if start_date <= int(d) <= end_date
    ]


## VIX-style calculation engine


In [8]:
# ============================================================
# Core VIX-style variance math
# ============================================================

def settlement_minutes_after_midnight_et(root):
    """
    Approximate VIX methodology settlement timing.

    SPX monthly options are AM-settled.
    SPXW weekly options are PM-settled.
    """
    if root == "SPX":
        return 9 * 60 + 30       # 9:30 AM ET

    if root == "SPXW":
        return 16 * 60           # 4:00 PM ET

    raise ValueError(f"Unknown root: {root}")


def minutes_to_expiry_vix_method(trade_date, exp_yyyymmdd, root, calc_time_ms):
    """
    Minutes from calculation time to option settlement time.
    """
    trade_dt = yyyymmdd_to_date(trade_date)
    exp_dt = yyyymmdd_to_date(exp_yyyymmdd)

    calc_minutes_after_midnight = int(calc_time_ms // 60000)
    settlement_minutes = settlement_minutes_after_midnight_et(root)

    days_diff = (exp_dt - trade_dt).days

    return days_diff * MINUTES_PER_DAY + settlement_minutes - calc_minutes_after_midnight


def _prepare_call_put_tables(chain):
    """
    Create call and put tables indexed by strike.
    """
    df = chain.copy()

    df["strike"] = pd.to_numeric(df["strike"], errors="coerce")
    df["bid"] = pd.to_numeric(df["bid"], errors="coerce")
    df["ask"] = pd.to_numeric(df["ask"], errors="coerce")
    df["mid"] = pd.to_numeric(df["mid"], errors="coerce")

    df = df.dropna(subset=["strike", "bid", "ask", "mid", "right"])
    df = df[df["ask"] >= 0]
    df = df[df["bid"] >= 0]

    calls = (
        df[df["right"] == "C"]
        .sort_values("strike")
        .drop_duplicates(subset=["strike"], keep="last")
        .set_index("strike")
    )

    puts = (
        df[df["right"] == "P"]
        .sort_values("strike")
        .drop_duplicates(subset=["strike"], keep="last")
        .set_index("strike")
    )

    return calls, puts


def _select_otm_options_with_bid_rule(options_df, ascending=True):
    """
    Select OTM options using a simplified Cboe-style nonzero-bid rule.

    Walk away from ATM.
    Keep options with positive bid.
    Stop after two consecutive zero-bid options.
    """
    options_df = options_df.sort_values("strike", ascending=ascending)

    selected_rows = []
    consecutive_zero_bids = 0

    for _, row in options_df.iterrows():
        if row["bid"] <= 0:
            consecutive_zero_bids += 1

            if consecutive_zero_bids >= 2:
                break

            continue

        consecutive_zero_bids = 0
        selected_rows.append(row)

    if len(selected_rows) == 0:
        return pd.DataFrame(columns=options_df.columns)

    return pd.DataFrame(selected_rows)


def calc_single_term_variance(chain, minutes_to_expiry, r=DEFAULT_RISK_FREE_RATE):
    """
    Calculate VIX-style annualized variance for one expiration.
    """
    T = minutes_to_expiry / MINUTES_PER_YEAR

    calls, puts = _prepare_call_put_tables(chain)

    common_strikes = sorted(set(calls.index).intersection(set(puts.index)))

    if len(common_strikes) == 0:
        raise ValueError("No common call/put strikes found")

    parity_rows = []

    for K in common_strikes:
        call_mid = calls.loc[K, "mid"]
        put_mid = puts.loc[K, "mid"]
        diff = abs(call_mid - put_mid)

        parity_rows.append({
            "strike": K,
            "call_mid": call_mid,
            "put_mid": put_mid,
            "abs_call_put_diff": diff,
        })

    parity_df = pd.DataFrame(parity_rows)
    min_row = parity_df.loc[parity_df["abs_call_put_diff"].idxmin()]

    K_star = float(min_row["strike"])
    call_mid_star = float(min_row["call_mid"])
    put_mid_star = float(min_row["put_mid"])

    F = K_star + math.exp(r * T) * (call_mid_star - put_mid_star)

    all_strikes = sorted(set(calls.index).union(set(puts.index)))
    strikes_below_or_equal_forward = [K for K in all_strikes if K <= F]

    if len(strikes_below_or_equal_forward) == 0:
        raise ValueError("Could not find K0 below forward")

    K0 = max(strikes_below_or_equal_forward)

    put_rows = []

    for K in sorted([x for x in puts.index if x < K0], reverse=True):
        row = puts.loc[K].copy()
        row["strike"] = K
        row["QK"] = row["mid"]
        put_rows.append(row)

    put_otm_raw = pd.DataFrame(put_rows)
    put_otm = _select_otm_options_with_bid_rule(put_otm_raw, ascending=False)

    call_rows = []

    for K in sorted([x for x in calls.index if x > K0]):
        row = calls.loc[K].copy()
        row["strike"] = K
        row["QK"] = row["mid"]
        call_rows.append(row)

    call_otm_raw = pd.DataFrame(call_rows)
    call_otm = _select_otm_options_with_bid_rule(call_otm_raw, ascending=True)

    if K0 not in calls.index or K0 not in puts.index:
        raise ValueError(f"K0={K0} missing call or put quote")

    k0_row = calls.loc[K0].copy()
    k0_row["strike"] = K0
    k0_row["QK"] = (calls.loc[K0, "mid"] + puts.loc[K0, "mid"]) / 2
    k0_row["bid"] = (calls.loc[K0, "bid"] + puts.loc[K0, "bid"]) / 2
    k0_row["ask"] = (calls.loc[K0, "ask"] + puts.loc[K0, "ask"]) / 2
    k0_row["right"] = "K0_AVG"

    selected_options = pd.concat(
        [
            put_otm,
            pd.DataFrame([k0_row]),
            call_otm,
        ],
        ignore_index=True,
    )

    selected_options = selected_options.sort_values("strike").reset_index(drop=True)

    strikes = selected_options["strike"].astype(float).values
    QK = selected_options["QK"].astype(float).values

    delta_K = np.zeros(len(strikes))

    for i in range(len(strikes)):
        if i == 0:
            delta_K[i] = strikes[i + 1] - strikes[i]
        elif i == len(strikes) - 1:
            delta_K[i] = strikes[i] - strikes[i - 1]
        else:
            delta_K[i] = (strikes[i + 1] - strikes[i - 1]) / 2

    selected_options["delta_K"] = delta_K

    selected_options["contribution"] = (
        selected_options["delta_K"]
        / (selected_options["strike"] ** 2)
        * math.exp(r * T)
        * selected_options["QK"]
    )

    variance = (
        (2 / T) * selected_options["contribution"].sum()
        - (1 / T) * ((F / K0) - 1) ** 2
    )

    return {
        "variance": variance,
        "F": F,
        "K0": K0,
        "K_star": K_star,
        "T": T,
        "minutes": minutes_to_expiry,
        "num_options": len(selected_options),
        "selected_options": selected_options,
    }


In [9]:
# ============================================================
# v0.7 expiration universe: Friday-cycle, holiday-adjusted
# ============================================================

def next_calendar_friday_after_date(dt):
    """
    Return the next calendar Friday on or after dt.

    Used to detect Friday-cycle expirations that are shifted earlier
    because the calendar Friday is a market holiday.
    """
    days_until_friday = (4 - dt.weekday()) % 7
    return dt + timedelta(days=days_until_friday)


def is_last_trading_day_before_closed_friday(exp_yyyymmdd, spx_trading_dates):
    """
    True if exp_yyyymmdd is the final SPX trading day before a closed calendar Friday.

    This handles Good Friday, July 4 on a Friday, Christmas on a Friday,
    and any other Friday market holiday.
    """
    exp_dt = yyyymmdd_to_date(exp_yyyymmdd)
    exp_int = int(exp_yyyymmdd)
    trading_date_set = set(int(x) for x in spx_trading_dates)

    if exp_int not in trading_date_set:
        return False

    if exp_dt.weekday() == 4:
        return False

    next_friday = next_calendar_friday_after_date(exp_dt)
    next_friday_int = int_date_from_date(next_friday)

    if next_friday_int in trading_date_set:
        return False

    check_dates = pd.date_range(
        start=exp_dt + timedelta(days=1),
        end=next_friday - timedelta(days=1),
        freq="D",
    )

    for d in check_dates:
        d_int = int(d.strftime("%Y%m%d"))
        if d_int in trading_date_set:
            return False

    return True


def is_friday_cycle_expiration_v7(exp_yyyymmdd, spx_trading_dates):
    """
    Friday-cycle expiration definition:

    1. Normal Friday expirations, but only if the expiration date is an SPX trading day.
    2. Non-Friday expirations that are the final SPX trading day
       before a closed calendar Friday.

    A closed Friday itself is not eligible.
    """
    exp_dt = yyyymmdd_to_date(exp_yyyymmdd)
    exp_int = int(exp_yyyymmdd)
    trading_date_set = set(int(x) for x in spx_trading_dates)

    if exp_int not in trading_date_set:
        return False

    if exp_dt.weekday() == 4:
        return True

    return is_last_trading_day_before_closed_friday(exp_yyyymmdd, spx_trading_dates)


def is_holiday_adjusted_monthly_expiration_v7(exp_yyyymmdd, spx_trading_dates):
    """
    True if this expiration is the holiday-adjusted version of a standard
    monthly third-Friday SPX expiration.
    """
    exp_dt = yyyymmdd_to_date(exp_yyyymmdd)

    if exp_dt.weekday() == 4:
        return False

    next_friday = next_calendar_friday_after_date(exp_dt)

    if not is_third_friday(next_friday):
        return False

    return is_last_trading_day_before_closed_friday(
        exp_yyyymmdd=exp_yyyymmdd,
        spx_trading_dates=spx_trading_dates,
    )


def preferred_root_for_expiration_v7(exp_yyyymmdd, spx_trading_dates):
    """
    Prefer SPX for standard monthly AM-settled expirations.

    Includes:
        - normal third-Friday SPX expirations
        - holiday-adjusted monthly SPX expirations shifted earlier

    Otherwise prefer SPXW for weekly/daily PM-settled options.
    """
    exp_dt = yyyymmdd_to_date(exp_yyyymmdd)

    if is_third_friday(exp_dt) and exp_yyyymmdd in spx_exps:
        return "SPX"

    if (
        is_holiday_adjusted_monthly_expiration_v7(exp_yyyymmdd, spx_trading_dates)
        and exp_yyyymmdd in spx_exps
    ):
        return "SPX"

    if exp_yyyymmdd in spxw_exps:
        return "SPXW"

    if exp_yyyymmdd in spx_exps:
        return "SPX"

    raise ValueError(f"Expiration {exp_yyyymmdd} not found in SPX or SPXW expiration lists")


def get_friday_cycle_expiration_candidates(
    trade_date,
    calc_time_ms=CALC_TIME_MS,
    min_minutes_to_expiry=1,
    spx_trading_dates=None,
):
    """
    Return future Friday-cycle SPX/SPXW expirations with days-to-expiry.

    Includes:
        - normal Friday expirations
        - holiday-adjusted expirations shifted earlier when Friday is closed
    """
    if spx_trading_dates is None:
        spx_trading_dates = load_spx_trading_dates()

    all_exps = sorted(set(spx_exps).union(set(spxw_exps)))
    rows = []

    for exp in all_exps:
        if not is_friday_cycle_expiration_v7(exp, spx_trading_dates):
            continue

        root = preferred_root_for_expiration_v7(exp, spx_trading_dates)

        minutes = minutes_to_expiry_vix_method(
            trade_date=trade_date,
            exp_yyyymmdd=exp,
            root=root,
            calc_time_ms=calc_time_ms,
        )

        if minutes <= min_minutes_to_expiry:
            continue

        rows.append({
            "root": root,
            "expiration": exp,
            "minutes": minutes,
            "days": minutes / MINUTES_PER_DAY,
        })

    candidates = pd.DataFrame(rows)

    if candidates.empty:
        raise ValueError(f"No future Friday-cycle expirations found for {trade_date}")

    return candidates.sort_values("minutes").reset_index(drop=True)


def choose_expiration_pair_for_target_days(
    trade_date,
    target_days,
    calc_time_ms=CALC_TIME_MS,
):
    """
    Choose the two Friday-cycle expirations that bracket a target tenor.
    """
    candidates = get_friday_cycle_expiration_candidates(
        trade_date=trade_date,
        calc_time_ms=calc_time_ms,
    )

    target_minutes = target_days * MINUTES_PER_DAY

    before = candidates[candidates["minutes"] <= target_minutes]
    after = candidates[candidates["minutes"] >= target_minutes]

    if before.empty:
        raise ValueError(f"No expiration before target {target_days}d for {trade_date}")

    if after.empty:
        raise ValueError(f"No expiration after target {target_days}d for {trade_date}")

    near_idx = before.index[-1]
    next_idx = after.index[0]

    if near_idx == next_idx:
        if next_idx + 1 < len(candidates):
            next_idx = next_idx + 1
        elif near_idx - 1 >= 0:
            near_idx = near_idx - 1
        else:
            raise ValueError(f"Could not form expiration pair for {target_days}d")

    near_exp = candidates.loc[near_idx].to_dict()
    next_exp = candidates.loc[next_idx].to_dict()

    return near_exp, next_exp


def get_required_chains_for_target_tenors(
    trade_date,
    target_tenor_days=TARGET_TENOR_DAYS,
    calc_time_ms=CALC_TIME_MS,
):
    """
    Return:
        required_by_tenor: one row per target tenor / near-next leg
        unique_chains: one row per unique option chain that must be pulled
    """
    rows = []

    for target_days in target_tenor_days:
        near_exp, next_exp = choose_expiration_pair_for_target_days(
            trade_date=trade_date,
            target_days=target_days,
            calc_time_ms=calc_time_ms,
        )

        rows.append({
            "target_days": target_days,
            "leg": "near",
            "root": near_exp["root"],
            "expiration": near_exp["expiration"],
            "minutes": near_exp["minutes"],
            "days": near_exp["days"],
        })

        rows.append({
            "target_days": target_days,
            "leg": "next",
            "root": next_exp["root"],
            "expiration": next_exp["expiration"],
            "minutes": next_exp["minutes"],
            "days": next_exp["days"],
        })

    required_by_tenor = pd.DataFrame(rows)

    unique_chains = (
        required_by_tenor[["root", "expiration", "minutes", "days"]]
        .drop_duplicates()
        .sort_values("minutes")
        .reset_index(drop=True)
    )

    return required_by_tenor, unique_chains


In [10]:
# ============================================================
# Raw option-chain cache and parallel loading
# ============================================================

def get_chain_cache_path(root, trade_date, expi, calc_time_ms):
    """
    Local cache file path for one option chain.
    """
    time_label = cache_time_label(calc_time_ms)
    filename = f"{root}_{int(trade_date)}_{int(expi)}_{time_label}.pkl"

    return CHAIN_CACHE_DIR / filename


def get_chain_at_time_cached(
    root,
    expi,
    trade_date,
    calc_time_ms,
    force_refresh=False,
    verbose=False,
):
    """
    Cached wrapper around get_chain_at_time().

    If the chain exists locally, load it.
    Otherwise pull from ThetaData and save it.
    """
    cache_path = get_chain_cache_path(
        root=root,
        trade_date=trade_date,
        expi=expi,
        calc_time_ms=calc_time_ms,
    )

    if cache_path.exists() and not force_refresh:
        if verbose:
            print(f"Loading from cache: {cache_path}")

        return pd.read_pickle(cache_path)

    if verbose:
        print(f"Pulling from ThetaData: {root} {trade_date} {expi}")

    chain = get_chain_at_time(
        root=root,
        expi=expi,
        trade_date=trade_date,
        calc_time_ms=calc_time_ms,
        verbose=verbose,
    )

    chain.to_pickle(cache_path)

    if verbose:
        print(f"Saved to cache: {cache_path}")

    return chain


def pull_unique_chains_parallel_cached(
    trade_date,
    unique_chains,
    calc_time_ms=CALC_TIME_MS,
    max_workers=4,
    force_refresh=False,
    verbose=False,
):
    """
    Pull/load each unique option chain once, in parallel.

    Uses local cache:
        if cached file exists -> load it
        else -> pull from ThetaData and save it
    """
    chain_results = {}

    def _pull_one_chain(row):
        root = row["root"]
        expiration = int(row["expiration"])

        chain = get_chain_at_time_cached(
            root=root,
            expi=expiration,
            trade_date=trade_date,
            calc_time_ms=calc_time_ms,
            force_refresh=force_refresh,
            verbose=verbose,
        )

        return (root, expiration), chain

    rows = [row for _, row in unique_chains.iterrows()]

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [
            executor.submit(_pull_one_chain, row)
            for row in rows
        ]

        for future in as_completed(futures):
            key, chain = future.result()
            chain_results[key] = chain

    return chain_results


In [11]:
# ============================================================
# Multi-tenor VIX-style term-structure engine
# ============================================================

def interpolate_variance_to_target_days(term_df, target_days):
    """
    Interpolate near and next term variances to a custom target tenor.

    Returns annualized variance in decimal form.
    Example:
        0.0190 means about 13.78 vol points, because sqrt(0.0190) * 100.
    """
    df = term_df.sort_values("minutes").reset_index(drop=True)

    if len(df) != 2:
        raise ValueError("term_df must have exactly two rows")

    N1 = df.loc[0, "minutes"]
    N2 = df.loc[1, "minutes"]

    T1 = N1 / MINUTES_PER_YEAR
    T2 = N2 / MINUTES_PER_YEAR

    var1 = df.loc[0, "variance"]
    var2 = df.loc[1, "variance"]

    target_minutes = target_days * MINUTES_PER_DAY

    if not (N1 <= target_minutes <= N2):
        raise ValueError(
            f"Target tenor {target_days} days is not bracketed by expirations: "
            f"near={N1 / MINUTES_PER_DAY:.2f} days, "
            f"next={N2 / MINUTES_PER_DAY:.2f} days"
        )

    interpolated_variance = (
        T1 * var1 * ((N2 - target_minutes) / (N2 - N1))
        + T2 * var2 * ((target_minutes - N1) / (N2 - N1))
    ) * (MINUTES_PER_YEAR / target_minutes)

    return interpolated_variance


def calculate_variance_for_unique_chains(unique_chains, chain_results, r):
    """
    Calculate one VIX-style variance for each unique option expiration.
    """
    rows = []
    calc_results = {}

    for _, row in unique_chains.iterrows():
        root = row["root"]
        expiration = int(row["expiration"])
        minutes = int(row["minutes"])
        days = float(row["days"])

        key = (root, expiration)
        chain = chain_results[key]

        calc = calc_single_term_variance(
            chain=chain,
            minutes_to_expiry=minutes,
            r=r,
        )

        calc_results[key] = calc

        rows.append({
            "root": root,
            "expiration": expiration,
            "minutes": minutes,
            "days": days,
            "variance": calc["variance"],
            "F": calc["F"],
            "K0": calc["K0"],
            "K_star": calc["K_star"],
            "num_options": calc["num_options"],
        })

    variance_table = pd.DataFrame(rows).sort_values("minutes").reset_index(drop=True)

    return variance_table, calc_results


def calculate_vix_term_structure_for_date_v7_cached(
    trade_date,
    target_tenor_days=TARGET_TENOR_DAYS,
    calc_time_ms=CALC_TIME_MS,
    rate_symbol=DEFAULT_RATE_SYMBOL,
    max_workers=4,
    force_refresh=False,
    return_details=True,
    verbose=False,
):
    """
    Calculate v0.7 VIX-style implied vol for multiple target tenors on one date.

    Uses:
        - same-date SOFR rate
        - Friday-cycle, holiday-adjusted SPX/SPXW expirations
        - parallel chain loading
        - local raw chain cache
    """
    rate = get_interest_rate_for_date_v3(
        symbol=rate_symbol,
        trade_date=trade_date,
    )

    required_by_tenor, unique_chains = get_required_chains_for_target_tenors(
        trade_date=trade_date,
        target_tenor_days=target_tenor_days,
        calc_time_ms=calc_time_ms,
    )

    chain_results = pull_unique_chains_parallel_cached(
        trade_date=trade_date,
        unique_chains=unique_chains,
        calc_time_ms=calc_time_ms,
        max_workers=max_workers,
        force_refresh=force_refresh,
        verbose=verbose,
    )

    variance_table, calc_results = calculate_variance_for_unique_chains(
        unique_chains=unique_chains,
        chain_results=chain_results,
        r=rate,
    )

    variance_lookup = {
        (row["root"], int(row["expiration"])): row
        for _, row in variance_table.iterrows()
    }

    output_rows = []

    for target_days in target_tenor_days:
        pair = required_by_tenor[
            required_by_tenor["target_days"] == target_days
        ].copy()

        if len(pair) != 2:
            raise ValueError(f"Expected two expiration rows for target {target_days}d")

        term_rows = []

        for _, leg_row in pair.iterrows():
            key = (leg_row["root"], int(leg_row["expiration"]))
            var_row = variance_lookup[key]

            term_rows.append({
                "term": leg_row["leg"],
                "root": leg_row["root"],
                "expiration": int(leg_row["expiration"]),
                "minutes": int(leg_row["minutes"]),
                "days": float(leg_row["days"]),
                "variance": float(var_row["variance"]),
                "F": float(var_row["F"]),
                "K0": float(var_row["K0"]),
                "num_options": int(var_row["num_options"]),
            })

        term_df = pd.DataFrame(term_rows).sort_values("minutes").reset_index(drop=True)

        implied_variance = interpolate_variance_to_target_days(
            term_df=term_df,
            target_days=target_days,
        )

        implied_vol = 100 * math.sqrt(implied_variance)

        output_rows.append({
            "trade_date": trade_date,
            "target_days": target_days,
            "rate_symbol": rate_symbol,
            "rate_pct": rate * 100,
            "implied_variance": implied_variance,
            "vix_style_vol": implied_vol,
            "near_root": term_df.loc[0, "root"],
            "near_expiration": term_df.loc[0, "expiration"],
            "near_days": term_df.loc[0, "days"],
            "near_variance": term_df.loc[0, "variance"],
            "next_root": term_df.loc[1, "root"],
            "next_expiration": term_df.loc[1, "expiration"],
            "next_days": term_df.loc[1, "days"],
            "next_variance": term_df.loc[1, "variance"],
        })

    result_df = pd.DataFrame(output_rows).sort_values("target_days").reset_index(drop=True)

    result = {
        "trade_date": trade_date,
        "rate_symbol": rate_symbol,
        "rate_decimal": rate,
        "rate_pct": rate * 100,
        "results_df": result_df,
    }

    if return_details:
        result["required_by_tenor"] = required_by_tenor
        result["unique_chains"] = unique_chains
        result["variance_table"] = variance_table
        result["chain_results"] = chain_results
        result["calc_results"] = calc_results

    return result


## Processed history and batch updates


In [12]:
# ============================================================
# Processed history load / save / upsert
# ============================================================

def load_existing_term_structure_history():
    """
    Load existing processed history if it exists.

    Prefer parquet if available, otherwise CSV.
    """
    if TERM_STRUCTURE_HISTORY_PARQUET.exists():
        try:
            return pd.read_parquet(TERM_STRUCTURE_HISTORY_PARQUET)
        except Exception as e:
            print(f"Could not read parquet file. Falling back to CSV if available. Error: {e}")

    if TERM_STRUCTURE_HISTORY_CSV.exists():
        return pd.read_csv(TERM_STRUCTURE_HISTORY_CSV)

    return pd.DataFrame()


def save_term_structure_history(history_df):
    """
    Save processed history.

    Always writes CSV.
    Tries to write parquet if the environment supports it.
    """
    history_df = history_df.sort_values(["trade_date", "target_days"]).reset_index(drop=True)

    history_df.to_csv(TERM_STRUCTURE_HISTORY_CSV, index=False)

    try:
        history_df.to_parquet(TERM_STRUCTURE_HISTORY_PARQUET, index=False)
        print(f"Saved CSV: {TERM_STRUCTURE_HISTORY_CSV}")
        print(f"Saved parquet: {TERM_STRUCTURE_HISTORY_PARQUET}")
    except Exception as e:
        print(f"Saved CSV: {TERM_STRUCTURE_HISTORY_CSV}")
        print(f"Could not save parquet. CSV is still saved. Error: {e}")


def upsert_term_structure_history(new_rows_df):
    """
    Append/replace processed history rows.

    Replacement key:
        trade_date + target_days + methodology_version + quote_time
    """
    existing_df = load_existing_term_structure_history()

    if new_rows_df.empty:
        print("No new rows to save.")
        return existing_df

    key_cols = [
        "trade_date",
        "target_days",
        "methodology_version",
        "quote_time",
    ]

    missing_cols = [c for c in key_cols if c not in new_rows_df.columns]

    if missing_cols:
        raise ValueError(f"new_rows_df is missing key columns: {missing_cols}")

    if existing_df.empty:
        updated_df = new_rows_df.copy()
    else:
        existing_key = existing_df[key_cols].astype(str).agg("|".join, axis=1)
        new_key = new_rows_df[key_cols].astype(str).agg("|".join, axis=1)

        existing_without_replaced = existing_df[~existing_key.isin(set(new_key))].copy()

        updated_df = pd.concat(
            [existing_without_replaced, new_rows_df],
            ignore_index=True,
        )

    updated_df = updated_df.sort_values(["trade_date", "target_days"]).reset_index(drop=True)

    save_term_structure_history(updated_df)

    print("Rows in updated history:", len(updated_df))
    print("Min date:", updated_df["trade_date"].min())
    print("Max date:", updated_df["trade_date"].max())

    return updated_df


In [13]:
# ============================================================
# Batch calculation and missing-date update
# ============================================================

def run_vix_term_structure_batch_v7(
    trade_dates,
    target_tenor_days=TARGET_TENOR_DAYS,
    calc_time_ms=CALC_TIME_MS,
    rate_symbol=DEFAULT_RATE_SYMBOL,
    max_workers=4,
    force_refresh=False,
    continue_on_error=True,
):
    """
    Calculate v0.7 multi-tenor VIX-style term structure for a list of trade dates.

    Returns:
        results_df: one row per trade_date / target_days
        errors_df: one row per failed trade_date, if any
    """
    all_results = []
    errors = []
    trade_dates = [int(d) for d in trade_dates]

    for i, trade_date in enumerate(trade_dates, start=1):
        print(f"[{i}/{len(trade_dates)}] Calculating {trade_date}...")

        try:
            result = calculate_vix_term_structure_for_date_v7_cached(
                trade_date=trade_date,
                target_tenor_days=target_tenor_days,
                calc_time_ms=calc_time_ms,
                rate_symbol=rate_symbol,
                max_workers=max_workers,
                force_refresh=force_refresh,
                return_details=False,
                verbose=False,
            )

            df = result["results_df"].copy()

            df["methodology_version"] = METHODOLOGY_VERSION
            df["expiration_universe"] = EXPIRATION_UNIVERSE
            df["trading_calendar_source"] = TRADING_CALENDAR_SOURCE
            df["rate_source"] = RATE_SOURCE
            df["option_source"] = OPTION_SOURCE
            df["quote_time"] = QUOTE_TIME_LABEL
            df["calc_time_ms"] = calc_time_ms

            all_results.append(df)

        except Exception as e:
            errors.append({
                "trade_date": trade_date,
                "error": str(e),
            })

            print(f"ERROR on {trade_date}: {e}")

            if not continue_on_error:
                raise

    if len(all_results) > 0:
        results_df = pd.concat(all_results, ignore_index=True)
        results_df = results_df.sort_values(["trade_date", "target_days"]).reset_index(drop=True)
    else:
        results_df = pd.DataFrame()

    errors_df = pd.DataFrame(errors)

    return results_df, errors_df


def get_completed_trade_dates_v7(history_df=None, target_tenor_days=TARGET_TENOR_DAYS):
    """
    Return trade dates that already have a complete set of target tenors
    for the v0.7 methodology version and quote time.
    """
    if history_df is None:
        history_df = load_existing_term_structure_history()

    if history_df.empty:
        return set()

    filtered = history_df[
        (history_df["methodology_version"] == METHODOLOGY_VERSION)
        & (history_df["quote_time"].astype(str) == str(QUOTE_TIME_LABEL))
    ].copy()

    if filtered.empty:
        return set()

    target_set = set(int(x) for x in target_tenor_days)
    completed_dates = set()

    for trade_date, group in filtered.groupby("trade_date"):
        tenor_set = set(int(x) for x in group["target_days"].unique())

        if target_set.issubset(tenor_set):
            completed_dates.add(int(trade_date))

    return completed_dates


def find_missing_trade_dates_v7(candidate_trade_dates, target_tenor_days=TARGET_TENOR_DAYS):
    """
    Given candidate trade dates, return only dates that do not yet have
    a complete v0.7 processed history.
    """
    history_df = load_existing_term_structure_history()

    completed_dates = get_completed_trade_dates_v7(
        history_df=history_df,
        target_tenor_days=target_tenor_days,
    )

    candidate_dates_clean = sorted(set(int(x) for x in candidate_trade_dates))

    return [
        d for d in candidate_dates_clean
        if d not in completed_dates
    ]


def update_term_structure_history_for_dates_v7(
    candidate_trade_dates,
    target_tenor_days=TARGET_TENOR_DAYS,
    calc_time_ms=CALC_TIME_MS,
    rate_symbol=DEFAULT_RATE_SYMBOL,
    max_workers=4,
    force_refresh=False,
    continue_on_error=True,
):
    """
    Update processed v0.7 term-structure history for missing dates only.
    """
    missing_dates = find_missing_trade_dates_v7(
        candidate_trade_dates=candidate_trade_dates,
        target_tenor_days=target_tenor_days,
    )

    print("Candidate dates:", len(set(candidate_trade_dates)))
    print("Missing/incomplete v0.7 dates:", len(missing_dates))

    if len(missing_dates) == 0:
        print("v0.7 history is already complete for the supplied dates.")
        return load_existing_term_structure_history(), pd.DataFrame()

    results_df, errors_df = run_vix_term_structure_batch_v7(
        trade_dates=missing_dates,
        target_tenor_days=target_tenor_days,
        calc_time_ms=calc_time_ms,
        rate_symbol=rate_symbol,
        max_workers=max_workers,
        force_refresh=force_refresh,
        continue_on_error=continue_on_error,
    )

    if not results_df.empty:
        updated_history_df = upsert_term_structure_history(results_df)
    else:
        updated_history_df = load_existing_term_structure_history()

    return updated_history_df, errors_df


def update_term_structure_history_for_range_v7(
    start_date,
    end_date,
    target_tenor_days=TARGET_TENOR_DAYS,
    calc_time_ms=CALC_TIME_MS,
    rate_symbol=DEFAULT_RATE_SYMBOL,
    max_workers=4,
    force_refresh=False,
    continue_on_error=True,
):
    """
    Update processed v0.7 history over an actual SPX trading-date range.
    """
    trade_dates = get_spx_trade_dates_between(
        start_date=start_date,
        end_date=end_date,
    )

    return update_term_structure_history_for_dates_v7(
        candidate_trade_dates=trade_dates,
        target_tenor_days=target_tenor_days,
        calc_time_ms=calc_time_ms,
        rate_symbol=rate_symbol,
        max_workers=max_workers,
        force_refresh=force_refresh,
        continue_on_error=continue_on_error,
    )


In [14]:
# ============================================================
# Validation and audit helpers
# ============================================================

def validate_history_range_v7(
    start_date,
    end_date,
    target_tenor_days=TARGET_TENOR_DAYS,
):
    """
    Validate processed history completeness for a date range.

    Returns:
        summary_df, bad_dates_df, duplicates_df
    """
    history_df = load_existing_term_structure_history()

    trade_dates = get_spx_trade_dates_between(
        start_date=start_date,
        end_date=end_date,
    )

    history_slice = history_df[
        (history_df["trade_date"] >= int(start_date))
        & (history_df["trade_date"] <= int(end_date))
        & (history_df["methodology_version"] == METHODOLOGY_VERSION)
        & (history_df["quote_time"].astype(str) == str(QUOTE_TIME_LABEL))
    ].copy()

    summary = {
        "start_date": int(start_date),
        "end_date": int(end_date),
        "expected_trade_dates": len(trade_dates),
        "actual_trade_dates": history_slice["trade_date"].nunique() if not history_slice.empty else 0,
        "expected_rows": len(trade_dates) * len(target_tenor_days),
        "actual_rows": len(history_slice),
    }

    summary_df = pd.DataFrame([summary])

    if history_slice.empty:
        bad_dates_df = pd.DataFrame({
            "trade_date": trade_dates,
            "rows": 0,
            "unique_tenors": 0,
            "min_tenor": np.nan,
            "max_tenor": np.nan,
        })
        duplicates_df = pd.DataFrame()
        return summary_df, bad_dates_df, duplicates_df

    counts = (
        history_slice
        .groupby("trade_date")
        .agg(
            rows=("target_days", "count"),
            unique_tenors=("target_days", "nunique"),
            min_tenor=("target_days", "min"),
            max_tenor=("target_days", "max"),
        )
        .reset_index()
    )

    expected_dates_df = pd.DataFrame({"trade_date": trade_dates})
    counts = expected_dates_df.merge(counts, on="trade_date", how="left").fillna({
        "rows": 0,
        "unique_tenors": 0,
    })

    bad_dates_df = counts[
        (counts["rows"] != len(target_tenor_days))
        | (counts["unique_tenors"] != len(target_tenor_days))
        | (counts["min_tenor"] != min(target_tenor_days))
        | (counts["max_tenor"] != max(target_tenor_days))
    ].copy()

    duplicate_check = (
        history_slice
        .groupby(["trade_date", "target_days", "methodology_version", "quote_time"])
        .size()
        .reset_index(name="count")
    )

    duplicates_df = duplicate_check[duplicate_check["count"] > 1].copy()

    return summary_df, bad_dates_df, duplicates_df


def audit_closed_friday_expirations(start_date, end_date, spx_trading_dates=None):
    """
    Find closed calendar Fridays and show the prior trading-day expiration handling.
    """
    if spx_trading_dates is None:
        spx_trading_dates = load_spx_trading_dates()

    trading_date_set = set(int(x) for x in spx_trading_dates)
    all_expirations = set(spx_exps).union(set(spxw_exps))

    start_ts = pd.to_datetime(str(int(start_date)), format="%Y%m%d")
    end_ts = pd.to_datetime(str(int(end_date)), format="%Y%m%d")

    all_fridays = pd.date_range(start=start_ts, end=end_ts, freq="W-FRI")

    rows = []

    for friday in all_fridays:
        friday_int = int(friday.strftime("%Y%m%d"))

        if friday_int in trading_date_set:
            continue

        prior_trading_dates = [d for d in trading_date_set if d < friday_int]

        if len(prior_trading_dates) == 0:
            continue

        adjusted_exp = max(prior_trading_dates)
        expiration_exists = adjusted_exp in all_expirations

        if expiration_exists:
            if adjusted_exp in spx_exps and adjusted_exp in spxw_exps:
                listed_roots = "SPX,SPXW"
            elif adjusted_exp in spx_exps:
                listed_roots = "SPX"
            else:
                listed_roots = "SPXW"
        else:
            listed_roots = None

        included_by_v7 = is_friday_cycle_expiration_v7(
            adjusted_exp,
            spx_trading_dates,
        )

        preferred_root = None

        if expiration_exists and included_by_v7:
            preferred_root = preferred_root_for_expiration_v7(
                adjusted_exp,
                spx_trading_dates,
            )

        rows.append({
            "closed_friday": friday_int,
            "adjusted_expiration": adjusted_exp,
            "adjusted_expiration_weekday": yyyymmdd_to_date(adjusted_exp).strftime("%A"),
            "expiration_exists": expiration_exists,
            "listed_roots": listed_roots,
            "included_by_v7": included_by_v7,
            "preferred_root": preferred_root,
        })

    return pd.DataFrame(rows)


## Usage cells

Run the definition cells above first. Then run these usage cells manually.

Recommended production sequence:

1. Build/refresh the trading calendar with a future buffer.
2. Load/refresh FRED SOFR.
3. Run the smoke test.
4. Run one backfill chunk at a time.
5. Validate that chunk before moving on.


In [15]:
# ============================================================
# Build or refresh full backfill trading calendar
# ============================================================

FULL_BACKFILL_START_DATE = 20180625
FULL_BACKFILL_END_DATE = 20260625
CALENDAR_END_DATE_WITH_BUFFER = 20261231

spx_trading_dates_full_df = update_spx_trading_dates_file(
    start_date=FULL_BACKFILL_START_DATE,
    end_date=CALENDAR_END_DATE_WITH_BUFFER,
    force_refresh=True,
)

spx_trading_dates_full = load_spx_trading_dates()

print("Trading dates loaded:", len(spx_trading_dates_full))
print("First:", min(spx_trading_dates_full))
print("Last:", max(spx_trading_dates_full))
display(spx_trading_dates_full_df.tail())


Saved SPX trading dates to: C:\Users\patri\vrp_project\data\processed\spx_trading_dates.csv
Calendar source: exchange_calendar_XNYS
Rows: 2142
Min date: 20180625
Max date: 20261231
Trading dates loaded: 2142
First: 20180625
Last: 20261231


,trade_date
2137,20261224
2138,20261228
2139,20261229
2140,20261230
2141,20261231


In [16]:
# ============================================================
# Load or refresh FRED SOFR history
# ============================================================

sofr_df = update_fred_sofr_history(force_refresh=False)
display(sofr_df.tail())


SOFR rows: 2055
First SOFR date: 20180403
Last SOFR date: 20260625
Saved to: C:\Users\patri\vrp_project\data\external\fred_sofr_history.csv


,trade_date,observation_date,rate_pct,rate_decimal
2050,20260618,2026-06-18,3.62,0.0362
2051,20260622,2026-06-22,3.61,0.0361
2052,20260623,2026-06-23,3.62,0.0362
2053,20260624,2026-06-24,3.62,0.0362
2054,20260625,2026-06-25,3.64,0.0364


In [17]:
# ============================================================
# Confirm active processed history status
# ============================================================

history_df = load_existing_term_structure_history()
print("Rows loaded:", len(history_df))

if not history_df.empty:
    display(history_df.tail())


Rows loaded: 1179


,trade_date,target_days,rate_symbol,rate_pct,implied_variance,vix_style_vol,near_root,near_expiration,near_days,near_variance,...,next_expiration,next_days,next_variance,methodology_version,expiration_universe,trading_calendar_source,rate_source,option_source,quote_time,calc_time_ms
1174,20181231,21,SOFR,3.0,0.069851,26.429311,SPX,20190118,17.729167,0.071888,...,20190125,25.0,0.068084,v0.7_exchange_calendar_fred_sofr_1600,friday_cycle_holiday_adjusted,exchange_calendar_XNYS,FRED_SOFR,THETADATA_SPX_SPXW,160000,57600000
1175,20181231,24,SOFR,3.0,0.068470,26.166821,SPX,20190118,17.729167,0.071888,...,20190125,25.0,0.068084,v0.7_exchange_calendar_fred_sofr_1600,friday_cycle_holiday_adjusted,exchange_calendar_XNYS,FRED_SOFR,THETADATA_SPX_SPXW,160000,57600000
1176,20181231,27,SOFR,3.0,0.067580,25.996164,SPXW,20190125,25.000000,0.068084,...,20190201,32.0,0.066596,v0.7_exchange_calendar_fred_sofr_1600,friday_cycle_holiday_adjusted,exchange_calendar_XNYS,FRED_SOFR,THETADATA_SPX_SPXW,160000,57600000
1177,20181231,30,SOFR,3.0,0.066951,25.874798,SPXW,20190125,25.000000,0.068084,...,20190201,32.0,0.066596,v0.7_exchange_calendar_fred_sofr_1600,friday_cycle_holiday_adjusted,exchange_calendar_XNYS,FRED_SOFR,THETADATA_SPX_SPXW,160000,57600000
1178,20181231,33,SOFR,3.0,0.066331,25.754902,SPXW,20190201,32.000000,0.066596,...,20190208,39.0,0.065027,v0.7_exchange_calendar_fred_sofr_1600,friday_cycle_holiday_adjusted,exchange_calendar_XNYS,FRED_SOFR,THETADATA_SPX_SPXW,160000,57600000


In [27]:
# ============================================================
# v0.7 smoke test: early and recent dates
# ============================================================

SMOKE_TEST_DATES = [20180625, FULL_BACKFILL_END_DATE]

for smoke_test_date in SMOKE_TEST_DATES:
    smoke_test_result = calculate_vix_term_structure_for_date_v7_cached(
        trade_date=smoke_test_date,
        target_tenor_days=TARGET_TENOR_DAYS,
        calc_time_ms=CALC_TIME_MS,
        rate_symbol=DEFAULT_RATE_SYMBOL,
        max_workers=4,
        force_refresh=False,
        return_details=True,
        verbose=False,
    )

    print("Smoke test date:", smoke_test_date)
    display(smoke_test_result["results_df"])


Smoke test date: 20180625


,trade_date,target_days,rate_symbol,rate_pct,implied_variance,vix_style_vol,near_root,near_expiration,near_days,near_variance,next_root,next_expiration,next_days,next_variance
0,20180625,9,SOFR,1.91,0.030097,17.348587,SPXW,20180629,4.000000,0.026702,SPXW,20180706,11.000000,0.030591
1,20180625,12,SOFR,1.91,0.030592,17.490509,SPXW,20180706,11.000000,0.030591,SPXW,20180713,18.000000,0.030594
2,20180625,15,SOFR,1.91,0.030593,17.490890,SPXW,20180706,11.000000,0.030591,SPXW,20180713,18.000000,0.030594
3,20180625,18,SOFR,1.91,0.030594,17.491144,SPXW,20180713,18.000000,0.030594,SPX,20180720,24.729167,0.030164
4,20180625,21,SOFR,1.91,0.030368,17.426522,SPXW,20180713,18.000000,0.030594,SPX,20180720,24.729167,0.030164
5,20180625,24,SOFR,1.91,0.030199,17.377898,SPXW,20180713,18.000000,0.030594,SPX,20180720,24.729167,0.030164
6,20180625,27,SOFR,1.91,0.030101,17.349740,SPX,20180720,24.729167,0.030164,SPXW,20180727,32.000000,0.029994
7,20180625,30,SOFR,1.91,0.030033,17.330003,SPX,20180720,24.729167,0.030164,SPXW,20180727,32.000000,0.029994
8,20180625,33,SOFR,1.91,0.030050,17.335065,SPXW,20180727,32.000000,0.029994,SPXW,20180803,39.000000,0.030326


Smoke test date: 20260625


,trade_date,target_days,rate_symbol,rate_pct,implied_variance,vix_style_vol,near_root,near_expiration,near_days,near_variance,next_root,next_expiration,next_days,next_variance
0,20260625,9,SOFR,3.64,0.032229,17.952376,SPXW,20260702,7.000000,0.032178,SPXW,20260710,15.000000,0.032300
1,20260625,12,SOFR,3.64,0.032273,17.964763,SPXW,20260702,7.000000,0.032178,SPXW,20260710,15.000000,0.032300
2,20260625,15,SOFR,3.64,0.032300,17.972192,SPXW,20260710,15.000000,0.032300,SPX,20260717,21.729167,0.030548
3,20260625,18,SOFR,3.64,0.031357,17.707876,SPXW,20260710,15.000000,0.032300,SPX,20260717,21.729167,0.030548
4,20260625,21,SOFR,3.64,0.030683,17.516638,SPXW,20260710,15.000000,0.032300,SPX,20260717,21.729167,0.030548
5,20260625,24,SOFR,3.64,0.032504,18.028765,SPX,20260717,21.729167,0.030548,SPXW,20260724,29.000000,0.035731
6,20260625,27,SOFR,3.64,0.034583,18.596577,SPX,20260717,21.729167,0.030548,SPXW,20260724,29.000000,0.035731
7,20260625,30,SOFR,3.64,0.036027,18.980878,SPXW,20260724,29.000000,0.035731,SPXW,20260731,36.000000,0.037462
8,20260625,33,SOFR,3.64,0.036810,19.185842,SPXW,20260724,29.000000,0.035731,SPXW,20260731,36.000000,0.037462


In [ ]:
# ============================================================
# Production backfill: one chunk at a time
# ============================================================

# Start with 2018 partial year. After validation passes, change these dates
# to 2019, 2020, 2021, etc.
BACKFILL_START_DATE = 20180625
BACKFILL_END_DATE = 20181231

history_df, errors_df = update_term_structure_history_for_range_v7(
    start_date=BACKFILL_START_DATE,
    end_date=BACKFILL_END_DATE,
    target_tenor_days=TARGET_TENOR_DAYS,
    calc_time_ms=CALC_TIME_MS,
    rate_symbol=DEFAULT_RATE_SYMBOL,
    max_workers=4,
    force_refresh=False,
    continue_on_error=True,
)

print("Rows in history:", len(history_df))
print("Errors:", len(errors_df))
display(errors_df)


In [ ]:
# ============================================================
# Validate processed history for the current backfill range
# ============================================================

summary_df, bad_dates_df, duplicates_df = validate_history_range_v7(
    start_date=BACKFILL_START_DATE,
    end_date=BACKFILL_END_DATE,
    target_tenor_days=TARGET_TENOR_DAYS,
)

display(summary_df)

print("Bad/incomplete dates:", len(bad_dates_df))
display(bad_dates_df)

print("Duplicate key rows:", len(duplicates_df))
display(duplicates_df)


## Suggested full backfill chunks

After the 2018 partial-year chunk validates, run the production backfill cell one year at a time:

- `20190101` to `20191231`
- `20200101` to `20201231`
- `20210101` to `20211231`
- `20220101` to `20221231`
- `20230101` to `20231231`
- `20240101` to `20241231`
- `20250101` to `20251231`
- `20260101` to `20260625`

Validate after each chunk before moving to the next one.


In [30]:
# ============================================================
# v0.7 backfill: 2018 partial year
# ============================================================

BACKFILL_START_DATE = 20180625
BACKFILL_END_DATE = 20181231

history_df, errors_df = update_term_structure_history_for_range_v7(
    start_date=BACKFILL_START_DATE,
    end_date=BACKFILL_END_DATE,
    target_tenor_days=TARGET_TENOR_DAYS,
    calc_time_ms=CALC_TIME_MS,
    rate_symbol=DEFAULT_RATE_SYMBOL,
    max_workers=4,
    force_refresh=False,
    continue_on_error=True,
)

print("Rows in history:", len(history_df))
print("Errors:", len(errors_df))
display(errors_df)

Candidate dates: 131
Missing/incomplete v0.7 dates: 129
[1/129] Calculating 20180627...
[2/129] Calculating 20180628...
[3/129] Calculating 20180629...
[4/129] Calculating 20180702...
[5/129] Calculating 20180703...
[6/129] Calculating 20180705...
[7/129] Calculating 20180706...
[8/129] Calculating 20180709...
[9/129] Calculating 20180710...
[10/129] Calculating 20180711...
[11/129] Calculating 20180712...
[12/129] Calculating 20180713...
[13/129] Calculating 20180716...
[14/129] Calculating 20180717...
[15/129] Calculating 20180718...
[16/129] Calculating 20180719...
[17/129] Calculating 20180720...
[18/129] Calculating 20180723...
[19/129] Calculating 20180724...
[20/129] Calculating 20180725...
[21/129] Calculating 20180726...
[22/129] Calculating 20180727...
[23/129] Calculating 20180730...
[24/129] Calculating 20180731...
[25/129] Calculating 20180801...
[26/129] Calculating 20180802...
[27/129] Calculating 20180803...
[28/129] Calculating 20180806...
[29/129] Calculating 20180807

""


In [31]:
# ============================================================
# Validate v0.7 2018 partial-year backfill
# ============================================================

history_check_df = load_existing_term_structure_history()

start_date = 20180625
end_date = 20181231

trade_dates_2018 = get_spx_trade_dates_between(
    start_date=start_date,
    end_date=end_date
)

slice_2018 = history_check_df[
    (history_check_df["trade_date"] >= start_date)
    & (history_check_df["trade_date"] <= end_date)
    & (history_check_df["methodology_version"] == METHODOLOGY_VERSION)
].copy()

print("2018 rows:", len(slice_2018))
print("2018 unique trade dates:", slice_2018["trade_date"].nunique())
print("Expected trade dates:", len(trade_dates_2018))
print("Expected rows:", len(trade_dates_2018) * len(TARGET_TENOR_DAYS))

counts_2018 = (
    slice_2018
    .groupby("trade_date")
    .agg(
        rows=("target_days", "count"),
        unique_tenors=("target_days", "nunique"),
        min_tenor=("target_days", "min"),
        max_tenor=("target_days", "max")
    )
    .reset_index()
)

bad_2018_dates = counts_2018[
    (counts_2018["rows"] != len(TARGET_TENOR_DAYS))
    | (counts_2018["unique_tenors"] != len(TARGET_TENOR_DAYS))
    | (counts_2018["min_tenor"] != min(TARGET_TENOR_DAYS))
    | (counts_2018["max_tenor"] != max(TARGET_TENOR_DAYS))
]

duplicate_check_2018 = (
    slice_2018
    .groupby(["trade_date", "target_days", "methodology_version", "quote_time"])
    .size()
    .reset_index(name="count")
)

duplicates_2018 = duplicate_check_2018[
    duplicate_check_2018["count"] > 1
]

print("Bad/incomplete 2018 dates:", len(bad_2018_dates))
display(bad_2018_dates)

print("Duplicate key rows:", len(duplicates_2018))
display(duplicates_2018)

2018 rows: 1179
2018 unique trade dates: 131
Expected trade dates: 131
Expected rows: 1179
Bad/incomplete 2018 dates: 0


,trade_date,rows,unique_tenors,min_tenor,max_tenor


Duplicate key rows: 0


,trade_date,target_days,methodology_version,quote_time,count


In [ ]:
# ============================================================
# v0.7 backfill: 2019-2020
# ============================================================

BACKFILL_START_DATE = 20190101
BACKFILL_END_DATE = 20201231

history_df, errors_df = update_term_structure_history_for_range_v7(
    start_date=BACKFILL_START_DATE,
    end_date=BACKFILL_END_DATE,
    target_tenor_days=TARGET_TENOR_DAYS,
    calc_time_ms=CALC_TIME_MS,
    rate_symbol=DEFAULT_RATE_SYMBOL,
    max_workers=4,
    force_refresh=False,
    continue_on_error=True,
)

print("Rows in history:", len(history_df))
print("Errors:", len(errors_df))
display(errors_df)

Candidate dates: 505
Missing/incomplete v0.7 dates: 505
[1/505] Calculating 20190102...
[2/505] Calculating 20190103...
[3/505] Calculating 20190104...
[4/505] Calculating 20190107...
[5/505] Calculating 20190108...
[6/505] Calculating 20190109...
[7/505] Calculating 20190110...
[8/505] Calculating 20190111...
[9/505] Calculating 20190114...
[10/505] Calculating 20190115...
[11/505] Calculating 20190116...
[12/505] Calculating 20190117...
[13/505] Calculating 20190118...
[14/505] Calculating 20190122...
[15/505] Calculating 20190123...
[16/505] Calculating 20190124...
[17/505] Calculating 20190125...
[18/505] Calculating 20190128...
[19/505] Calculating 20190129...
[20/505] Calculating 20190130...
[21/505] Calculating 20190131...
[22/505] Calculating 20190201...
[23/505] Calculating 20190204...
[24/505] Calculating 20190205...
[25/505] Calculating 20190206...
[26/505] Calculating 20190207...
[27/505] Calculating 20190208...
[28/505] Calculating 20190211...
[29/505] Calculating 20190212

In [28]:
# ============================================================
# Check current saved history after interrupted run
# ============================================================

history_df = load_existing_term_structure_history()

print("Rows saved:", len(history_df))
print("Min date:", history_df["trade_date"].min() if len(history_df) else None)
print("Max date:", history_df["trade_date"].max() if len(history_df) else None)

date_counts = (
    history_df
    .groupby("trade_date")
    .agg(rows=("target_days", "count"))
    .reset_index()
    .sort_values("trade_date")
)

display(date_counts.tail(20))

Rows saved: 1179
Min date: 20180625
Max date: 20181231


,trade_date,rows
111,20181130,9
112,20181203,9
113,20181204,9
114,20181206,9
115,20181207,9
116,20181210,9
117,20181211,9
118,20181212,9
119,20181213,9
120,20181214,9


In [18]:
# ============================================================
# v0.7 backfill: 2019-2020 retry after sleep interruption
# ============================================================

BACKFILL_START_DATE = 20190101
BACKFILL_END_DATE = 20201231

history_df, errors_df = update_term_structure_history_for_range_v7(
    start_date=BACKFILL_START_DATE,
    end_date=BACKFILL_END_DATE,
    target_tenor_days=TARGET_TENOR_DAYS,
    calc_time_ms=CALC_TIME_MS,
    rate_symbol=DEFAULT_RATE_SYMBOL,
    max_workers=4,
    force_refresh=False,
    continue_on_error=True,
)

print("Rows in history:", len(history_df))
print("Errors:", len(errors_df))
display(errors_df)

Candidate dates: 505
Missing/incomplete v0.7 dates: 505
[1/505] Calculating 20190102...
[2/505] Calculating 20190103...
[3/505] Calculating 20190104...
[4/505] Calculating 20190107...
[5/505] Calculating 20190108...
[6/505] Calculating 20190109...
[7/505] Calculating 20190110...
[8/505] Calculating 20190111...
[9/505] Calculating 20190114...
[10/505] Calculating 20190115...
[11/505] Calculating 20190116...
[12/505] Calculating 20190117...
[13/505] Calculating 20190118...
[14/505] Calculating 20190122...
[15/505] Calculating 20190123...
[16/505] Calculating 20190124...
[17/505] Calculating 20190125...
[18/505] Calculating 20190128...
[19/505] Calculating 20190129...
[20/505] Calculating 20190130...
[21/505] Calculating 20190131...
[22/505] Calculating 20190201...
[23/505] Calculating 20190204...
[24/505] Calculating 20190205...
[25/505] Calculating 20190206...
[26/505] Calculating 20190207...
[27/505] Calculating 20190208...
[28/505] Calculating 20190211...
[29/505] Calculating 20190212

,trade_date,error
0,20200903,"HTTPConnectionPool(host='127.0.0.1', port=2550..."
1,20201127,'strike'
2,20201224,'strike'


In [19]:
# ============================================================
# Inspect 2019-2020 errors
# ============================================================

print("Errors:", len(errors_df))
display(errors_df)

if len(errors_df):
    print("Failed dates:")
    display(errors_df["trade_date"].drop_duplicates().sort_values().tolist())

Errors: 3


,trade_date,error
0,20200903,"HTTPConnectionPool(host='127.0.0.1', port=2550..."
1,20201127,'strike'
2,20201224,'strike'


Failed dates:


[20200903, 20201127, 20201224]

In [20]:
# ============================================================
# Retry failed dates only
# ============================================================

failed_dates = (
    errors_df["trade_date"]
    .drop_duplicates()
    .sort_values()
    .astype(int)
    .tolist()
)

for retry_date in failed_dates:
    print("=" * 80)
    print("Retrying:", retry_date)

    history_df, retry_errors_df = update_term_structure_history_for_range_v7(
        start_date=retry_date,
        end_date=retry_date,
        target_tenor_days=TARGET_TENOR_DAYS,
        calc_time_ms=CALC_TIME_MS,
        rate_symbol=DEFAULT_RATE_SYMBOL,
        max_workers=4,
        force_refresh=True,
        continue_on_error=True,
    )

    print("Rows in history:", len(history_df))
    print("Retry errors:", len(retry_errors_df))
    display(retry_errors_df)

Retrying: 20200903
Candidate dates: 1
Missing/incomplete v0.7 dates: 1
[1/1] Calculating 20200903...
Saved CSV: C:\Users\patri\vrp_project\data\processed\vix_term_structure_history.csv
Saved parquet: C:\Users\patri\vrp_project\data\processed\vix_term_structure_history.parquet
Rows in updated history: 5706
Min date: 20180625
Max date: 20201231
Rows in history: 5706
Retry errors: 0


""


Retrying: 20201127
Candidate dates: 1
Missing/incomplete v0.7 dates: 1
[1/1] Calculating 20201127...
ERROR on 20201127: 'strike'
Rows in history: 5706
Retry errors: 1


,trade_date,error
0,20201127,'strike'


Retrying: 20201224
Candidate dates: 1
Missing/incomplete v0.7 dates: 1
[1/1] Calculating 20201224...
ERROR on 20201224: 'strike'
Rows in history: 5706
Retry errors: 1


,trade_date,error
0,20201224,'strike'


In [21]:
# ============================================================
# v0.7 market-close patch
# Normal days use 16:00 ET; early-close days use actual market close
# ============================================================

import pandas_market_calendars as mcal

METHODOLOGY_VERSION = "v0.7_exchange_calendar_fred_sofr_market_close"
QUOTE_TIME_LABEL = "market_close"

TRADING_CALENDAR_SOURCE = "exchange_calendar_XNYS"
RATE_SOURCE = "FRED_SOFR"
OPTION_SOURCE = "THETADATA_SPX_SPXW"


def get_market_close_time_for_trade_date(trade_date, calendar_name="XNYS"):
    """
    Return actual market close time for a trade date.

    Returns:
        close_time_label: "16:00" on normal days, "13:00" on early-close days
        close_time_ms: milliseconds after midnight ET
    """
    cal = mcal.get_calendar(calendar_name)

    date_iso = pd.to_datetime(str(int(trade_date)), format="%Y%m%d").strftime("%Y-%m-%d")

    schedule = cal.schedule(
        start_date=date_iso,
        end_date=date_iso
    )

    if schedule.empty:
        raise RuntimeError(f"No market calendar schedule found for {trade_date}")

    close_ts = schedule.iloc[0]["market_close"]

    # Calendar close is timezone-aware, usually UTC. Convert to New York time.
    close_ts_et = close_ts.tz_convert("America/New_York")

    close_time_label = close_ts_et.strftime("%H:%M")

    close_time_ms = (
        close_ts_et.hour * 60 * 60 * 1000
        + close_ts_et.minute * 60 * 1000
        + close_ts_et.second * 1000
    )

    return close_time_label, int(close_time_ms)


# Keep a reference to the fixed-time calculator
_calculate_vix_term_structure_for_date_v7_cached_fixed_time = (
    calculate_vix_term_structure_for_date_v7_cached
)


def calculate_vix_term_structure_for_date_v7_cached(
    trade_date,
    target_tenor_days=TARGET_TENOR_DAYS,
    calc_time_ms=CALC_TIME_MS,
    rate_symbol=DEFAULT_RATE_SYMBOL,
    max_workers=4,
    force_refresh=False,
    return_details=True,
    verbose=False,
):
    """
    Market-close wrapper around the v0.7 calculator.

    Normal days use 16:00 ET.
    Early-close days use the exchange-calendar close, usually 13:00 ET.
    """
    close_time_label, actual_calc_time_ms = get_market_close_time_for_trade_date(
        trade_date=trade_date
    )

    result = _calculate_vix_term_structure_for_date_v7_cached_fixed_time(
        trade_date=trade_date,
        target_tenor_days=target_tenor_days,
        calc_time_ms=actual_calc_time_ms,
        rate_symbol=rate_symbol,
        max_workers=max_workers,
        force_refresh=force_refresh,
        return_details=return_details,
        verbose=verbose,
    )

    # Add audit columns if result contains a dataframe.
    if isinstance(result, dict) and "results_df" in result:
        result["results_df"] = result["results_df"].copy()
        result["results_df"]["market_close_time"] = close_time_label
        result["results_df"]["calc_time_ms"] = actual_calc_time_ms

    elif isinstance(result, pd.DataFrame):
        result = result.copy()
        result["market_close_time"] = close_time_label
        result["calc_time_ms"] = actual_calc_time_ms

    return result


print("Methodology version:", METHODOLOGY_VERSION)
print("Quote time label:", QUOTE_TIME_LABEL)

for test_date in [20201125, 20201127, 20201224, 20201228]:
    close_label, close_ms = get_market_close_time_for_trade_date(test_date)
    print(test_date, close_label, close_ms)

Methodology version: v0.7_exchange_calendar_fred_sofr_market_close
Quote time label: market_close
20201125 16:00 57600000
20201127 13:00 46800000
20201224 13:00 46800000
20201228 16:00 57600000


In [22]:
# ============================================================
# Migrate existing v0.7 rows to market-close methodology label
# ============================================================

processed_history_csv = PROCESSED_DATA_DIR / "vix_term_structure_history.csv"
processed_history_parquet = PROCESSED_DATA_DIR / "vix_term_structure_history.parquet"

history_df = load_existing_term_structure_history()

print("Rows before migration:", len(history_df))

if len(history_df) > 0:
    history_df = history_df.copy()

    history_df["methodology_version"] = METHODOLOGY_VERSION
    history_df["quote_time"] = QUOTE_TIME_LABEL

    close_info = {
        int(date): get_market_close_time_for_trade_date(int(date))
        for date in history_df["trade_date"].drop_duplicates()
    }

    history_df["market_close_time"] = history_df["trade_date"].astype(int).map(
        lambda d: close_info[d][0]
    )

    history_df["calc_time_ms"] = history_df["trade_date"].astype(int).map(
        lambda d: close_info[d][1]
    )

    history_df.to_csv(processed_history_csv, index=False)
    history_df.to_parquet(processed_history_parquet, index=False)

    print("Saved migrated CSV:", processed_history_csv)
    print("Saved migrated parquet:", processed_history_parquet)

history_df = load_existing_term_structure_history()

print("Rows after migration:", len(history_df))
print("Methodology versions:")
display(history_df["methodology_version"].value_counts())

print("Quote time labels:")
display(history_df["quote_time"].value_counts())

Rows before migration: 5706
Saved migrated CSV: C:\Users\patri\vrp_project\data\processed\vix_term_structure_history.csv
Saved migrated parquet: C:\Users\patri\vrp_project\data\processed\vix_term_structure_history.parquet
Rows after migration: 5706
Methodology versions:


methodology_version
v0.7_exchange_calendar_fred_sofr_market_close    5706
Name: count, dtype: int64

Quote time labels:


quote_time
market_close    5706
Name: count, dtype: int64

In [23]:
# ============================================================
# Retry early-close failed dates using market-close time
# ============================================================

for retry_date in [20201127, 20201224]:
    print("=" * 80)
    print("Retrying:", retry_date)

    history_df, retry_errors_df = update_term_structure_history_for_range_v7(
        start_date=retry_date,
        end_date=retry_date,
        target_tenor_days=TARGET_TENOR_DAYS,
        calc_time_ms=CALC_TIME_MS,  # ignored by wrapper; market close is used
        rate_symbol=DEFAULT_RATE_SYMBOL,
        max_workers=4,
        force_refresh=True,
        continue_on_error=True,
    )

    print("Rows in history:", len(history_df))
    print("Retry errors:", len(retry_errors_df))
    display(retry_errors_df)

Retrying: 20201127
Candidate dates: 1
Missing/incomplete v0.7 dates: 1
[1/1] Calculating 20201127...
Saved CSV: C:\Users\patri\vrp_project\data\processed\vix_term_structure_history.csv
Saved parquet: C:\Users\patri\vrp_project\data\processed\vix_term_structure_history.parquet
Rows in updated history: 5715
Min date: 20180625
Max date: 20201231
Rows in history: 5715
Retry errors: 0


""


Retrying: 20201224
Candidate dates: 1
Missing/incomplete v0.7 dates: 1
[1/1] Calculating 20201224...
Saved CSV: C:\Users\patri\vrp_project\data\processed\vix_term_structure_history.csv
Saved parquet: C:\Users\patri\vrp_project\data\processed\vix_term_structure_history.parquet
Rows in updated history: 5724
Min date: 20180625
Max date: 20201231
Rows in history: 5724
Retry errors: 0


""


In [24]:
# ============================================================
# Validate v0.7 market-close 2019-2020 backfill
# ============================================================

history_check_df = load_existing_term_structure_history()

start_date = 20190101
end_date = 20201231

trade_dates_range = get_spx_trade_dates_between(
    start_date=start_date,
    end_date=end_date
)

slice_range = history_check_df[
    (history_check_df["trade_date"] >= start_date)
    & (history_check_df["trade_date"] <= end_date)
    & (history_check_df["methodology_version"] == METHODOLOGY_VERSION)
].copy()

print("Rows:", len(slice_range))
print("Unique trade dates:", slice_range["trade_date"].nunique())
print("Expected trade dates:", len(trade_dates_range))
print("Expected rows:", len(trade_dates_range) * len(TARGET_TENOR_DAYS))

counts_range = (
    slice_range
    .groupby("trade_date")
    .agg(
        rows=("target_days", "count"),
        unique_tenors=("target_days", "nunique"),
        min_tenor=("target_days", "min"),
        max_tenor=("target_days", "max")
    )
    .reset_index()
)

bad_dates = counts_range[
    (counts_range["rows"] != len(TARGET_TENOR_DAYS))
    | (counts_range["unique_tenors"] != len(TARGET_TENOR_DAYS))
    | (counts_range["min_tenor"] != min(TARGET_TENOR_DAYS))
    | (counts_range["max_tenor"] != max(TARGET_TENOR_DAYS))
]

duplicate_check = (
    slice_range
    .groupby(["trade_date", "target_days", "methodology_version", "quote_time"])
    .size()
    .reset_index(name="count")
)

duplicates = duplicate_check[duplicate_check["count"] > 1]

print("Bad/incomplete dates:", len(bad_dates))
display(bad_dates)

print("Duplicate key rows:", len(duplicates))
display(duplicates)

print("Market-close times in this range:")
display(slice_range["market_close_time"].value_counts())

Rows: 4545
Unique trade dates: 505
Expected trade dates: 505
Expected rows: 4545
Bad/incomplete dates: 0


,trade_date,rows,unique_tenors,min_tenor,max_tenor


Duplicate key rows: 0


,trade_date,target_days,methodology_version,quote_time,count


Market-close times in this range:


market_close_time
16:00    4500
13:00      45
Name: count, dtype: int64

In [25]:
# ============================================================
# v0.7 market-close backfill: 2021-2022
# ============================================================

BACKFILL_START_DATE = 20210101
BACKFILL_END_DATE = 20221231

history_df, errors_df = update_term_structure_history_for_range_v7(
    start_date=BACKFILL_START_DATE,
    end_date=BACKFILL_END_DATE,
    target_tenor_days=TARGET_TENOR_DAYS,
    calc_time_ms=CALC_TIME_MS,  # market-close wrapper overrides this per date
    rate_symbol=DEFAULT_RATE_SYMBOL,
    max_workers=4,
    force_refresh=False,
    continue_on_error=True,
)

print("Rows in history:", len(history_df))
print("Errors:", len(errors_df))
display(errors_df)

Candidate dates: 503
Missing/incomplete v0.7 dates: 503
[1/503] Calculating 20210104...
[2/503] Calculating 20210105...
[3/503] Calculating 20210106...
[4/503] Calculating 20210107...
[5/503] Calculating 20210108...
[6/503] Calculating 20210111...
[7/503] Calculating 20210112...
[8/503] Calculating 20210113...
[9/503] Calculating 20210114...
[10/503] Calculating 20210115...
[11/503] Calculating 20210119...
[12/503] Calculating 20210120...
[13/503] Calculating 20210121...
[14/503] Calculating 20210122...
[15/503] Calculating 20210125...
[16/503] Calculating 20210126...
[17/503] Calculating 20210127...
[18/503] Calculating 20210128...
[19/503] Calculating 20210129...
[20/503] Calculating 20210201...
[21/503] Calculating 20210202...
[22/503] Calculating 20210203...
[23/503] Calculating 20210204...
[24/503] Calculating 20210205...
[25/503] Calculating 20210208...
[26/503] Calculating 20210209...
[27/503] Calculating 20210210...
[28/503] Calculating 20210211...
[29/503] Calculating 20210212

,trade_date,error
0,20220222,'strike'
1,20220331,index 1 is out of bounds for axis 0 with size 1
2,20220531,Could not find K0 below forward


In [26]:
# ============================================================
# Retry failed 2021-2022 dates
# ============================================================

failed_dates = [20220222, 20220331, 20220531]

for retry_date in failed_dates:
    print("=" * 80)
    print("Retrying:", retry_date)

    history_df, retry_errors_df = update_term_structure_history_for_range_v7(
        start_date=retry_date,
        end_date=retry_date,
        target_tenor_days=TARGET_TENOR_DAYS,
        calc_time_ms=CALC_TIME_MS,  # market-close wrapper overrides this
        rate_symbol=DEFAULT_RATE_SYMBOL,
        max_workers=4,
        force_refresh=True,
        continue_on_error=True,
    )

    print("Rows in history:", len(history_df))
    print("Retry errors:", len(retry_errors_df))
    display(retry_errors_df)

Retrying: 20220222
Candidate dates: 1
Missing/incomplete v0.7 dates: 1
[1/1] Calculating 20220222...
ERROR on 20220222: 'strike'
Rows in history: 10224
Retry errors: 1


,trade_date,error
0,20220222,'strike'


Retrying: 20220331
Candidate dates: 1
Missing/incomplete v0.7 dates: 1
[1/1] Calculating 20220331...
ERROR on 20220331: index 1 is out of bounds for axis 0 with size 1
Rows in history: 10224
Retry errors: 1


,trade_date,error
0,20220331,index 1 is out of bounds for axis 0 with size 1


Retrying: 20220531
Candidate dates: 1
Missing/incomplete v0.7 dates: 1
[1/1] Calculating 20220531...
ERROR on 20220531: Could not find K0 below forward
Rows in history: 10224
Retry errors: 1


,trade_date,error
0,20220531,Could not find K0 below forward


In [27]:
# ============================================================
# Diagnostic: test failed dates using quote times before close
# Does not save results
# ============================================================

def time_label_to_ms(time_label):
    """
    Convert HH:MM string to milliseconds after midnight.
    """
    hour, minute = [int(x) for x in time_label.split(":")]
    return (hour * 60 * 60 * 1000) + (minute * 60 * 1000)


failed_dates = [20220222, 20220331, 20220531]

test_time_labels = [
    "16:00",
    "15:59",
    "15:58",
    "15:55",
    "15:50",
]

diagnostic_rows = []

for test_date in failed_dates:
    print("=" * 80)
    print("Testing date:", test_date)

    for time_label in test_time_labels:
        test_ms = time_label_to_ms(time_label)

        print(f"  Trying {time_label}...")

        try:
            result = _calculate_vix_term_structure_for_date_v7_cached_fixed_time(
                trade_date=test_date,
                target_tenor_days=TARGET_TENOR_DAYS,
                calc_time_ms=test_ms,
                rate_symbol=DEFAULT_RATE_SYMBOL,
                max_workers=1,
                force_refresh=True,
                return_details=True,
                verbose=False,
            )

            results_df = result["results_df"]

            print(f"    SUCCESS: rows={len(results_df)}")

            diagnostic_rows.append({
                "trade_date": test_date,
                "time_label": time_label,
                "status": "success",
                "rows": len(results_df),
                "error": None,
            })

            # Stop testing earlier times once this date works.
            break

        except Exception as e:
            print(f"    Failed: {e}")

            diagnostic_rows.append({
                "trade_date": test_date,
                "time_label": time_label,
                "status": "failed",
                "rows": None,
                "error": str(e),
            })

diagnostic_df = pd.DataFrame(diagnostic_rows)
display(diagnostic_df)

Testing date: 20220222
  Trying 16:00...
    Failed: 'strike'
  Trying 15:59...
    Failed: 'strike'
  Trying 15:58...
    Failed: 'strike'
  Trying 15:55...
    Failed: 'strike'
  Trying 15:50...
    Failed: 'strike'
Testing date: 20220331
  Trying 16:00...
    Failed: index 1 is out of bounds for axis 0 with size 1
  Trying 15:59...
    SUCCESS: rows=9
Testing date: 20220531
  Trying 16:00...
    Failed: Could not find K0 below forward
  Trying 15:59...
    SUCCESS: rows=9


,trade_date,time_label,status,rows,error
0,20220222,16:00,failed,NaN,'strike'
1,20220222,15:59,failed,NaN,'strike'
2,20220222,15:58,failed,NaN,'strike'
3,20220222,15:55,failed,NaN,'strike'
4,20220222,15:50,failed,NaN,'strike'
5,20220331,16:00,failed,NaN,index 1 is out of bounds for axis 0 with size 1
6,20220331,15:59,success,9.0,None
7,20220531,16:00,failed,NaN,Could not find K0 below forward
8,20220531,15:59,success,9.0,None


In [28]:
# ============================================================
# v0.7 market-close fallback patch
# Try exact market close, then slightly before close if needed
# ============================================================

def ms_to_time_label(ms):
    """
    Convert milliseconds after midnight to HH:MM.
    """
    total_minutes = int(ms // (60 * 1000))
    hour = total_minutes // 60
    minute = total_minutes % 60
    return f"{hour:02d}:{minute:02d}"


def calculate_vix_term_structure_for_date_v7_cached(
    trade_date,
    target_tenor_days=TARGET_TENOR_DAYS,
    calc_time_ms=CALC_TIME_MS,
    rate_symbol=DEFAULT_RATE_SYMBOL,
    max_workers=4,
    force_refresh=False,
    return_details=True,
    verbose=False,
):
    """
    Market-close wrapper with quote-time fallback.

    Normal target:
        actual market close, usually 16:00 or 13:00 on early-close days

    Fallback:
        if exact close fails, try 1, 2, 5, then 10 minutes before close.

    Audit columns:
        market_close_time = official exchange close time
        quote_time_used = actual option quote time used
        quote_time_offset_minutes = minutes before close
        calc_time_ms = actual milliseconds-after-midnight used
    """
    market_close_label, market_close_ms = get_market_close_time_for_trade_date(
        trade_date=trade_date
    )

    fallback_minutes = [0, 1, 2, 5, 10]

    last_error = None

    for offset_minutes in fallback_minutes:
        actual_calc_time_ms = market_close_ms - offset_minutes * 60 * 1000
        quote_time_used = ms_to_time_label(actual_calc_time_ms)

        if verbose:
            print(
                f"Trying {trade_date} at {quote_time_used} "
                f"({offset_minutes} min before close)"
            )

        try:
            result = _calculate_vix_term_structure_for_date_v7_cached_fixed_time(
                trade_date=trade_date,
                target_tenor_days=target_tenor_days,
                calc_time_ms=actual_calc_time_ms,
                rate_symbol=rate_symbol,
                max_workers=max_workers,
                force_refresh=force_refresh,
                return_details=return_details,
                verbose=verbose,
            )

            if isinstance(result, dict) and "results_df" in result:
                result["results_df"] = result["results_df"].copy()
                result["results_df"]["market_close_time"] = market_close_label
                result["results_df"]["quote_time_used"] = quote_time_used
                result["results_df"]["quote_time_offset_minutes"] = offset_minutes
                result["results_df"]["calc_time_ms"] = actual_calc_time_ms

            elif isinstance(result, pd.DataFrame):
                result = result.copy()
                result["market_close_time"] = market_close_label
                result["quote_time_used"] = quote_time_used
                result["quote_time_offset_minutes"] = offset_minutes
                result["calc_time_ms"] = actual_calc_time_ms

            if verbose and offset_minutes > 0:
                print(f"Fallback succeeded for {trade_date} at {quote_time_used}")

            return result

        except Exception as e:
            last_error = e

            if verbose:
                print(f"Failed at {quote_time_used}: {e}")

    raise last_error

In [29]:
# ============================================================
# Retry dates recoverable with 15:59 fallback
# ============================================================

for retry_date in [20220331, 20220531]:
    print("=" * 80)
    print("Retrying:", retry_date)

    history_df, retry_errors_df = update_term_structure_history_for_range_v7(
        start_date=retry_date,
        end_date=retry_date,
        target_tenor_days=TARGET_TENOR_DAYS,
        calc_time_ms=CALC_TIME_MS,
        rate_symbol=DEFAULT_RATE_SYMBOL,
        max_workers=4,
        force_refresh=True,
        continue_on_error=True,
    )

    print("Rows in history:", len(history_df))
    print("Retry errors:", len(retry_errors_df))
    display(retry_errors_df)

Retrying: 20220331
Candidate dates: 1
Missing/incomplete v0.7 dates: 1
[1/1] Calculating 20220331...
Saved CSV: C:\Users\patri\vrp_project\data\processed\vix_term_structure_history.csv
Saved parquet: C:\Users\patri\vrp_project\data\processed\vix_term_structure_history.parquet
Rows in updated history: 10233
Min date: 20180625
Max date: 20221230
Rows in history: 10233
Retry errors: 0


""


Retrying: 20220531
Candidate dates: 1
Missing/incomplete v0.7 dates: 1
[1/1] Calculating 20220531...
Saved CSV: C:\Users\patri\vrp_project\data\processed\vix_term_structure_history.csv
Saved parquet: C:\Users\patri\vrp_project\data\processed\vix_term_structure_history.parquet
Rows in updated history: 10242
Min date: 20180625
Max date: 20221230
Rows in history: 10242
Retry errors: 0


""


In [30]:
# ============================================================
# Diagnose remaining failed date: 2022-02-22
# Identify which required chain is causing the issue
# ============================================================

diagnostic_date = 20220222

market_close_label, market_close_ms = get_market_close_time_for_trade_date(diagnostic_date)

print("Diagnostic date:", diagnostic_date)
print("Market close:", market_close_label, market_close_ms)

required_by_tenor, unique_chains = get_required_chains_for_target_tenors(
    trade_date=diagnostic_date,
    target_tenor_days=TARGET_TENOR_DAYS,
    calc_time_ms=market_close_ms,
)

print("\nRequired chains by tenor:")
for tenor, chains in required_by_tenor.items():
    print(f"{tenor}d:", chains)

print("\nUnique chains:")
display(pd.DataFrame(unique_chains))

Diagnostic date: 20220222
Market close: 16:00 57600000

Required chains by tenor:
target_daysd: 0      9
1      9
2     12
3     12
4     15
5     15
6     18
7     18
8     21
9     21
10    24
11    24
12    27
13    27
14    30
15    30
16    33
17    33
Name: target_days, dtype: int64
legd: 0     near
1     next
2     near
3     next
4     near
5     next
6     near
7     next
8     near
9     next
10    near
11    next
12    near
13    next
14    near
15    next
16    near
17    next
Name: leg, dtype: object
rootd: 0     SPXW
1     SPXW
2     SPXW
3     SPXW
4     SPXW
5     SPXW
6     SPXW
7      SPX
8     SPXW
9      SPX
10     SPX
11    SPXW
12     SPX
13    SPXW
14     SPX
15    SPXW
16    SPXW
17    SPXW
Name: root, dtype: object
expirationd: 0     20220225
1     20220304
2     20220304
3     20220311
4     20220304
5     20220311
6     20220311
7     20220318
8     20220311
9     20220318
10    20220318
11    20220325
12    20220318
13    20220325
14    20220318
15    202203

,root,expiration,minutes,days
0,SPXW,20220225,4320,3.000000
1,SPXW,20220304,14400,10.000000
2,SPXW,20220311,24480,17.000000
3,SPX,20220318,34170,23.729167
4,SPXW,20220325,44640,31.000000
5,SPXW,20220401,54720,38.000000


In [31]:
# ============================================================
# Pull each required chain one at a time
# ============================================================

chain_diagnostic_rows = []

for chain in unique_chains:
    root = chain["root"]
    expiration = chain["expiration"]

    print("=" * 80)
    print("Testing chain:", root, expiration)

    try:
        chain_df = get_option_chain_snapshot_v3_cached(
            root=root,
            trade_date=diagnostic_date,
            expiration=expiration,
            calc_time_ms=market_close_ms,
            force_refresh=True,
            verbose=False,
        )

        print("Rows:", len(chain_df))
        print("Columns:", list(chain_df.columns))

        chain_diagnostic_rows.append({
            "root": root,
            "expiration": expiration,
            "status": "success",
            "rows": len(chain_df),
            "columns": ", ".join(chain_df.columns),
            "error": None,
        })

    except Exception as e:
        print("ERROR:", e)

        chain_diagnostic_rows.append({
            "root": root,
            "expiration": expiration,
            "status": "failed",
            "rows": None,
            "columns": None,
            "error": str(e),
        })

chain_diagnostic_df = pd.DataFrame(chain_diagnostic_rows)
display(chain_diagnostic_df)

TypeError: string indices must be integers, not 'str'

In [32]:
# ============================================================
# Inspect required_by_tenor and unique_chains structure
# ============================================================

print("type(required_by_tenor):", type(required_by_tenor))
print("type(unique_chains):", type(unique_chains))

print("\nrequired_by_tenor:")
for k, v in required_by_tenor.items():
    print(k, type(v), repr(v))

print("\nunique_chains:")
for i, chain in enumerate(list(unique_chains)):
    print(i, type(chain), repr(chain))

print("\nAvailable chain-related functions:")
for name in sorted([name for name in globals().keys() if "chain" in name.lower()]):
    print(name)

type(required_by_tenor): <class 'pandas.core.frame.DataFrame'>
type(unique_chains): <class 'pandas.core.frame.DataFrame'>

required_by_tenor:
target_days <class 'pandas.core.series.Series'> 0      9
1      9
2     12
3     12
4     15
5     15
6     18
7     18
8     21
9     21
10    24
11    24
12    27
13    27
14    30
15    30
16    33
17    33
Name: target_days, dtype: int64
leg <class 'pandas.core.series.Series'> 0     near
1     next
2     near
3     next
4     near
5     next
6     near
7     next
8     near
9     next
10    near
11    next
12    near
13    next
14    near
15    next
16    near
17    next
Name: leg, dtype: object
root <class 'pandas.core.series.Series'> 0     SPXW
1     SPXW
2     SPXW
3     SPXW
4     SPXW
5     SPXW
6     SPXW
7      SPX
8     SPXW
9      SPX
10     SPX
11    SPXW
12     SPX
13    SPXW
14     SPX
15    SPXW
16    SPXW
17    SPXW
Name: root, dtype: object
expiration <class 'pandas.core.series.Series'> 0     20220225
1     20220304
2     20220

In [33]:
# ============================================================
# Pull each required chain one at a time - robust version
# ============================================================

def normalize_chain_entry(chain):
    """
    Convert chain entry into {"root": ..., "expiration": ...}.
    Handles common structures: dict, tuple/list, or string.
    """
    if isinstance(chain, dict):
        return {
            "root": chain.get("root"),
            "expiration": int(chain.get("expiration")),
        }

    if isinstance(chain, (tuple, list)):
        # Common case: ("SPXW", 20220318)
        if len(chain) >= 2:
            return {
                "root": chain[0],
                "expiration": int(chain[1]),
            }

    if isinstance(chain, str):
        # Try common formats like "SPXW_20220318" or "SPXW|20220318"
        for sep in ["_", "|", ",", "-"]:
            parts = chain.split(sep)
            for i, part in enumerate(parts):
                if part in ["SPX", "SPXW"] and i + 1 < len(parts):
                    possible_exp = "".join(ch for ch in parts[i + 1] if ch.isdigit())
                    if len(possible_exp) == 8:
                        return {
                            "root": part,
                            "expiration": int(possible_exp),
                        }

        raise ValueError(f"Could not parse chain string: {chain}")

    raise ValueError(f"Unsupported chain entry type: {type(chain)} / {chain}")


normalized_chains = []

for chain in unique_chains:
    normalized_chains.append(normalize_chain_entry(chain))

normalized_chains_df = (
    pd.DataFrame(normalized_chains)
    .drop_duplicates()
    .sort_values(["expiration", "root"])
    .reset_index(drop=True)
)

display(normalized_chains_df)

ValueError: Could not parse chain string: root

In [34]:
# ============================================================
# Normalize unique_chains when it is already a DataFrame
# ============================================================

print("type(unique_chains):", type(unique_chains))

if isinstance(unique_chains, pd.DataFrame):
    normalized_chains_df = unique_chains.copy()
else:
    normalized_chains_df = pd.DataFrame(unique_chains)

display(normalized_chains_df)
print("Columns:", list(normalized_chains_df.columns))

type(unique_chains): <class 'pandas.core.frame.DataFrame'>


,root,expiration,minutes,days
0,SPXW,20220225,4320,3.000000
1,SPXW,20220304,14400,10.000000
2,SPXW,20220311,24480,17.000000
3,SPX,20220318,34170,23.729167
4,SPXW,20220325,44640,31.000000
5,SPXW,20220401,54720,38.000000


Columns: ['root', 'expiration', 'minutes', 'days']


In [35]:
# ============================================================
# Test each required chain one at a time
# ============================================================

chain_diagnostic_rows = []

for _, row in normalized_chains_df.iterrows():
    root = row["root"]
    expiration = int(row["expiration"])

    print("=" * 80)
    print("Testing chain:", root, expiration)

    try:
        chain_df = get_option_chain_snapshot_v3_cached(
            root=root,
            trade_date=diagnostic_date,
            expiration=expiration,
            calc_time_ms=market_close_ms,
            force_refresh=True,
            verbose=False,
        )

        print("Rows:", len(chain_df))
        print("Columns:", list(chain_df.columns))

        chain_diagnostic_rows.append({
            "root": root,
            "expiration": expiration,
            "status": "success",
            "rows": len(chain_df),
            "columns": ", ".join(chain_df.columns),
            "error": None,
        })

    except Exception as e:
        print("ERROR:", e)

        chain_diagnostic_rows.append({
            "root": root,
            "expiration": expiration,
            "status": "failed",
            "rows": None,
            "columns": None,
            "error": str(e),
        })

chain_diagnostic_df = pd.DataFrame(chain_diagnostic_rows)
display(chain_diagnostic_df)

Testing chain: SPXW 20220225
ERROR: name 'get_option_chain_snapshot_v3_cached' is not defined
Testing chain: SPXW 20220304
ERROR: name 'get_option_chain_snapshot_v3_cached' is not defined
Testing chain: SPXW 20220311
ERROR: name 'get_option_chain_snapshot_v3_cached' is not defined
Testing chain: SPX 20220318
ERROR: name 'get_option_chain_snapshot_v3_cached' is not defined
Testing chain: SPXW 20220325
ERROR: name 'get_option_chain_snapshot_v3_cached' is not defined
Testing chain: SPXW 20220401
ERROR: name 'get_option_chain_snapshot_v3_cached' is not defined


,root,expiration,status,rows,columns,error
0,SPXW,20220225,failed,None,None,name 'get_option_chain_snapshot_v3_cached' is ...
1,SPXW,20220304,failed,None,None,name 'get_option_chain_snapshot_v3_cached' is ...
2,SPXW,20220311,failed,None,None,name 'get_option_chain_snapshot_v3_cached' is ...
3,SPX,20220318,failed,None,None,name 'get_option_chain_snapshot_v3_cached' is ...
4,SPXW,20220325,failed,None,None,name 'get_option_chain_snapshot_v3_cached' is ...
5,SPXW,20220401,failed,None,None,name 'get_option_chain_snapshot_v3_cached' is ...


In [36]:
# ============================================================
# Test each required chain one at a time - corrected function name
# ============================================================

chain_diagnostic_rows = []

for _, row in normalized_chains_df.iterrows():
    root = row["root"]
    expiration = int(row["expiration"])

    print("=" * 80)
    print("Testing chain:", root, expiration)

    try:
        chain_df = get_chain_at_time_cached(
            root=root,
            expi=expiration,
            trade_date=diagnostic_date,
            calc_time_ms=market_close_ms,
            force_refresh=True,
            verbose=False,
        )

        print("Rows:", len(chain_df))
        print("Columns:", list(chain_df.columns))

        chain_diagnostic_rows.append({
            "root": root,
            "expiration": expiration,
            "status": "success",
            "rows": len(chain_df),
            "columns": ", ".join(chain_df.columns),
            "error": None,
        })

    except Exception as e:
        print("ERROR:", e)

        chain_diagnostic_rows.append({
            "root": root,
            "expiration": expiration,
            "status": "failed",
            "rows": None,
            "columns": None,
            "error": str(e),
        })

chain_diagnostic_df = pd.DataFrame(chain_diagnostic_rows)
display(chain_diagnostic_df)

Testing chain: SPXW 20220225
Rows: 508
Columns: ['root', 'expiration', 'strike', 'right', 'bid', 'ask', 'mid', 'bid_size', 'ask_size', 'bid_exchange', 'ask_exchange', 'bid_condition', 'ask_condition', 'timestamp']
Testing chain: SPXW 20220304
Rows: 486
Columns: ['root', 'expiration', 'strike', 'right', 'bid', 'ask', 'mid', 'bid_size', 'ask_size', 'bid_exchange', 'ask_exchange', 'bid_condition', 'ask_condition', 'timestamp']
Testing chain: SPXW 20220311
Rows: 406
Columns: ['root', 'expiration', 'strike', 'right', 'bid', 'ask', 'mid', 'bid_size', 'ask_size', 'bid_exchange', 'ask_exchange', 'bid_condition', 'ask_condition', 'timestamp']
Testing chain: SPX 20220318
Rows: 836
Columns: ['root', 'expiration', 'strike', 'right', 'bid', 'ask', 'mid', 'bid_size', 'ask_size', 'bid_exchange', 'ask_exchange', 'bid_condition', 'ask_condition', 'timestamp']
Testing chain: SPXW 20220325
Rows: 362
Columns: ['root', 'expiration', 'strike', 'right', 'bid', 'ask', 'mid', 'bid_size', 'ask_size', 'bid_excha

,root,expiration,status,rows,columns,error
0,SPXW,20220225,success,508,"root, expiration, strike, right, bid, ask, mid...",None
1,SPXW,20220304,success,486,"root, expiration, strike, right, bid, ask, mid...",None
2,SPXW,20220311,success,406,"root, expiration, strike, right, bid, ask, mid...",None
3,SPX,20220318,success,836,"root, expiration, strike, right, bid, ask, mid...",None
4,SPXW,20220325,success,362,"root, expiration, strike, right, bid, ask, mid...",None
5,SPXW,20220401,success,246,"root, expiration, strike, right, bid, ask, mid...",None


In [37]:
# ============================================================
# Retry 2022-02-22 using refreshed chain cache
# ============================================================

retry_date = 20220222

history_df, retry_errors_df = update_term_structure_history_for_range_v7(
    start_date=retry_date,
    end_date=retry_date,
    target_tenor_days=TARGET_TENOR_DAYS,
    calc_time_ms=CALC_TIME_MS,
    rate_symbol=DEFAULT_RATE_SYMBOL,
    max_workers=1,
    force_refresh=False,
    continue_on_error=True,
)

print("Rows in history:", len(history_df))
print("Retry errors:", len(retry_errors_df))
display(retry_errors_df)

Candidate dates: 1
Missing/incomplete v0.7 dates: 1
[1/1] Calculating 20220222...
ERROR on 20220222: 'strike'
Rows in history: 10242
Retry errors: 1


,trade_date,error
0,20220222,'strike'


In [38]:
# ============================================================
# Full traceback for 2022-02-22
# Do not save; just expose where 'strike' happens
# ============================================================

retry_date = 20220222

result_20220222 = calculate_vix_term_structure_for_date_v7_cached(
    trade_date=retry_date,
    target_tenor_days=TARGET_TENOR_DAYS,
    calc_time_ms=CALC_TIME_MS,
    rate_symbol=DEFAULT_RATE_SYMBOL,
    max_workers=1,
    force_refresh=False,
    return_details=True,
    verbose=True,
)

display(result_20220222["results_df"])

Trying 20220222 at 16:00 (0 min before close)
Loading from cache: C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20220222_20220225_160000.pkl
Loading from cache: C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20220222_20220304_160000.pkl
Loading from cache: C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20220222_20220311_160000.pkl
Loading from cache: C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPX_20220222_20220318_160000.pkl
Loading from cache: C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20220222_20220325_160000.pkl
Loading from cache: C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20220222_20220401_160000.pkl
Failed at 16:00: 'strike'
Trying 20220222 at 15:59 (1 min before close)
Loading from cache: C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20220222_20220225_155900.pkl
Loading from cache: C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20220222_20220304_155900.pkl
Loading from cache: C:\User

KeyError: 'strike'

In [39]:
# ============================================================
# Diagnose which 2022-02-22 chain fails inside variance calc
# ============================================================

diagnostic_date = 20220222
market_close_label, market_close_ms = get_market_close_time_for_trade_date(diagnostic_date)

required_by_tenor, unique_chains = get_required_chains_for_target_tenors(
    trade_date=diagnostic_date,
    target_tenor_days=TARGET_TENOR_DAYS,
    calc_time_ms=market_close_ms,
)

if isinstance(unique_chains, pd.DataFrame):
    chains_to_test = unique_chains.copy()
else:
    chains_to_test = pd.DataFrame(unique_chains)

display(chains_to_test)

variance_diagnostic_rows = []

for _, row in chains_to_test.iterrows():
    root = row["root"]
    expiration = int(row["expiration"])

    # Try to get minutes column robustly
    possible_minute_cols = [
        "minutes_to_expiry",
        "minutes",
        "expiration_minutes",
        "mins_to_expiry"
    ]
    minutes_col = None
    for col in possible_minute_cols:
        if col in chains_to_test.columns:
            minutes_col = col
            break

    if minutes_col is None:
        print("Could not find minutes-to-expiry column. Columns are:")
        print(list(chains_to_test.columns))
        break

    minutes_to_expiry = float(row[minutes_col])

    print("=" * 80)
    print("Testing variance calc:", root, expiration, "minutes:", minutes_to_expiry)

    try:
        chain_df = get_chain_at_time_cached(
            root=root,
            expi=expiration,
            trade_date=diagnostic_date,
            calc_time_ms=market_close_ms,
            force_refresh=False,
            verbose=False,
        )

        print("Chain rows:", len(chain_df))
        print("Strike min/max:", chain_df["strike"].min(), chain_df["strike"].max())
        print("Rights:", chain_df["right"].value_counts().to_dict())

        calc = calc_single_term_variance(
            chain=chain_df,
            minutes_to_expiry=minutes_to_expiry,
            r=get_interest_rate_for_date_v3(DEFAULT_RATE_SYMBOL, diagnostic_date),
        )

        print("SUCCESS")

        variance_diagnostic_rows.append({
            "root": root,
            "expiration": expiration,
            "minutes_to_expiry": minutes_to_expiry,
            "status": "success",
            "error": None,
        })

    except Exception as e:
        print("FAILED:", repr(e))

        variance_diagnostic_rows.append({
            "root": root,
            "expiration": expiration,
            "minutes_to_expiry": minutes_to_expiry,
            "status": "failed",
            "error": repr(e),
        })

variance_diagnostic_df = pd.DataFrame(variance_diagnostic_rows)
display(variance_diagnostic_df)

,root,expiration,minutes,days
0,SPXW,20220225,4320,3.000000
1,SPXW,20220304,14400,10.000000
2,SPXW,20220311,24480,17.000000
3,SPX,20220318,34170,23.729167
4,SPXW,20220325,44640,31.000000
5,SPXW,20220401,54720,38.000000


Testing variance calc: SPXW 20220225 minutes: 4320.0
Chain rows: 508
Strike min/max: 1000.0 5800.0
Rights: {'C': 254, 'P': 254}
SUCCESS
Testing variance calc: SPXW 20220304 minutes: 14400.0
Chain rows: 486
Strike min/max: 800.0 5800.0
Rights: {'P': 243, 'C': 243}
SUCCESS
Testing variance calc: SPXW 20220311 minutes: 24480.0
Chain rows: 406
Strike min/max: 1200.0 5800.0
Rights: {'P': 203, 'C': 203}
SUCCESS
Testing variance calc: SPX 20220318 minutes: 34170.0
Chain rows: 836
Strike min/max: 200.0 6900.0
Rights: {'P': 418, 'C': 418}
SUCCESS
Testing variance calc: SPXW 20220325 minutes: 44640.0
Chain rows: 362
Strike min/max: 1200.0 5800.0
Rights: {'C': 181, 'P': 181}
FAILED: KeyError('strike')
Testing variance calc: SPXW 20220401 minutes: 54720.0
Chain rows: 246
Strike min/max: 1800.0 5600.0
Rights: {'P': 123, 'C': 123}
SUCCESS


,root,expiration,minutes_to_expiry,status,error
0,SPXW,20220225,4320.0,success,None
1,SPXW,20220304,14400.0,success,None
2,SPXW,20220311,24480.0,success,None
3,SPX,20220318,34170.0,success,None
4,SPXW,20220325,44640.0,failed,KeyError('strike')
5,SPXW,20220401,54720.0,success,None


In [40]:
# ============================================================
# Diagnose forward / K0 for failing chain
# 2022-02-22 SPXW 2022-03-25
# ============================================================

debug_date = 20220222
debug_root = "SPXW"
debug_expiration = 20220325
debug_minutes = 44640.0

debug_chain = get_chain_at_time_cached(
    root=debug_root,
    expi=debug_expiration,
    trade_date=debug_date,
    calc_time_ms=market_close_ms,
    force_refresh=False,
    verbose=False,
)

debug_rate = get_interest_rate_for_date_v3(DEFAULT_RATE_SYMBOL, debug_date)
debug_T = debug_minutes / MINUTES_PER_YEAR

print("Rows:", len(debug_chain))
print("Rate:", debug_rate)
print("T:", debug_T)
print("Rights:", debug_chain["right"].value_counts().to_dict())
print("Strike min/max:", debug_chain["strike"].min(), debug_chain["strike"].max())

# Pair calls and puts by strike
calls = (
    debug_chain[debug_chain["right"] == "C"]
    [["strike", "bid", "ask", "mid"]]
    .rename(columns={
        "bid": "call_bid",
        "ask": "call_ask",
        "mid": "call_mid"
    })
)

puts = (
    debug_chain[debug_chain["right"] == "P"]
    [["strike", "bid", "ask", "mid"]]
    .rename(columns={
        "bid": "put_bid",
        "ask": "put_ask",
        "mid": "put_mid"
    })
)

paired = calls.merge(puts, on="strike", how="inner")

paired["abs_call_put_diff"] = (paired["call_mid"] - paired["put_mid"]).abs()
paired["forward_candidate"] = (
    paired["strike"]
    + np.exp(debug_rate * debug_T) * (paired["call_mid"] - paired["put_mid"])
)

paired_sorted = paired.sort_values("abs_call_put_diff").reset_index(drop=True)

print("Paired strikes:", len(paired_sorted))
print("Top parity candidates:")
display(paired_sorted.head(20))

best = paired_sorted.iloc[0]

debug_forward = float(best["forward_candidate"])
debug_k0 = paired[paired["strike"] <= debug_forward]["strike"].max()

print("Best parity strike:", best["strike"])
print("Call mid:", best["call_mid"])
print("Put mid:", best["put_mid"])
print("Forward:", debug_forward)
print("K0:", debug_k0)

print("Puts <= K0:", len(debug_chain[(debug_chain["right"] == "P") & (debug_chain["strike"] <= debug_k0)]))
print("Calls >= K0:", len(debug_chain[(debug_chain["right"] == "C") & (debug_chain["strike"] >= debug_k0)]))

print("Nearest strikes around forward:")
display(
    paired[
        (paired["strike"] >= debug_forward - 300)
        & (paired["strike"] <= debug_forward + 300)
    ].sort_values("strike")
)

Rows: 362
Rate: 0.0005
T: 0.08493150684931507
Rights: {'C': 181, 'P': 181}
Strike min/max: 1200.0 5800.0
Paired strikes: 181
Top parity candidates:


,strike,call_bid,call_ask,call_mid,put_bid,put_ask,put_mid,abs_call_put_diff,forward_candidate
0,1200.0,0.0,0.0,0.00,0.0,0.0,0.00,0.00,1200.000000
1,4300.0,127.5,128.3,127.90,127.5,128.3,127.90,0.00,4300.000000
2,1400.0,0.0,0.0,0.00,0.0,0.0,0.00,0.00,1400.000000
3,4305.0,124.5,125.4,124.95,129.5,130.4,129.95,5.00,4299.999788
4,4295.0,130.5,131.3,130.90,125.5,126.3,125.90,5.00,4300.000212
5,4290.0,133.5,134.3,133.90,123.5,124.4,123.95,9.95,4299.950423
6,4310.0,121.6,122.4,122.00,131.6,132.4,132.00,10.00,4299.999575
7,4315.0,118.7,119.5,119.10,133.6,134.5,134.05,14.95,4300.049365
8,4285.0,136.6,137.4,137.00,121.6,122.4,122.00,15.00,4300.000637
9,4280.0,139.7,140.5,140.10,119.7,120.5,120.10,20.00,4300.000849


Best parity strike: 1200.0
Call mid: 0.0
Put mid: 0.0
Forward: 1200.0
K0: 1200.0
Puts <= K0: 1
Calls >= K0: 181
Nearest strikes around forward:


,strike,call_bid,call_ask,call_mid,put_bid,put_ask,put_mid,abs_call_put_diff,forward_candidate
17,1200.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1200.0
176,1400.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1400.0


In [41]:
# ============================================================
# v0.7 robust forward/K0 patch
# Exclude junk zero/zero parity candidates
# ============================================================

def _select_otm_options_with_bid_rule(options_df, ascending=True):
    """
    Select OTM options using a simplified Cboe-style nonzero-bid rule.

    Handles empty input safely.
    """
    if options_df is None or len(options_df) == 0:
        return pd.DataFrame(columns=["strike", "bid", "ask", "mid", "QK"])

    if "strike" not in options_df.columns:
        return pd.DataFrame(columns=list(options_df.columns) + ["strike"])

    options_df = options_df.sort_values("strike", ascending=ascending)

    selected_rows = []
    consecutive_zero_bids = 0

    for _, row in options_df.iterrows():
        if row["bid"] <= 0:
            consecutive_zero_bids += 1

            if consecutive_zero_bids >= 2:
                break

            continue

        consecutive_zero_bids = 0
        selected_rows.append(row)

    if len(selected_rows) == 0:
        return pd.DataFrame(columns=options_df.columns)

    return pd.DataFrame(selected_rows)


def calc_single_term_variance(chain, minutes_to_expiry, r=DEFAULT_RISK_FREE_RATE):
    """
    Calculate VIX-style annualized variance for one expiration.

    Robust parity improvement:
        Exclude call/put parity candidates where either side has zero mid.
        This avoids selecting junk zero/zero deep-strike quotes as the forward.
    """
    T = minutes_to_expiry / MINUTES_PER_YEAR

    calls, puts = _prepare_call_put_tables(chain)

    common_strikes = sorted(set(calls.index).intersection(set(puts.index)))

    if len(common_strikes) == 0:
        raise ValueError("No common call/put strikes found")

    parity_rows = []

    for K in common_strikes:
        call_mid = float(calls.loc[K, "mid"])
        put_mid = float(puts.loc[K, "mid"])
        call_bid = float(calls.loc[K, "bid"])
        put_bid = float(puts.loc[K, "bid"])
        call_ask = float(calls.loc[K, "ask"])
        put_ask = float(puts.loc[K, "ask"])

        parity_rows.append({
            "strike": K,
            "call_mid": call_mid,
            "put_mid": put_mid,
            "call_bid": call_bid,
            "put_bid": put_bid,
            "call_ask": call_ask,
            "put_ask": put_ask,
            "abs_call_put_diff": abs(call_mid - put_mid),
        })

    parity_df = pd.DataFrame(parity_rows)

    # Robust filter: avoid junk zero/zero quotes.
    parity_candidates = parity_df[
        (parity_df["call_mid"] > 0)
        & (parity_df["put_mid"] > 0)
        & (parity_df["call_ask"] >= parity_df["call_bid"])
        & (parity_df["put_ask"] >= parity_df["put_bid"])
    ].copy()

    if parity_candidates.empty:
        raise ValueError("No valid parity candidates after filtering zero/invalid quotes")

    min_row = parity_candidates.loc[
        parity_candidates["abs_call_put_diff"].idxmin()
    ]

    K_star = float(min_row["strike"])
    call_mid_star = float(min_row["call_mid"])
    put_mid_star = float(min_row["put_mid"])

    F = K_star + math.exp(r * T) * (call_mid_star - put_mid_star)

    all_strikes = sorted(set(calls.index).union(set(puts.index)))
    strikes_below_or_equal_forward = [K for K in all_strikes if K <= F]

    if len(strikes_below_or_equal_forward) == 0:
        raise ValueError("Could not find K0 below forward")

    K0 = max(strikes_below_or_equal_forward)

    put_rows = []

    for K in sorted([x for x in puts.index if x < K0], reverse=True):
        row = puts.loc[K].copy()
        row["strike"] = K
        row["QK"] = row["mid"]
        put_rows.append(row)

    put_otm_raw = pd.DataFrame(put_rows)
    put_otm = _select_otm_options_with_bid_rule(put_otm_raw, ascending=False)

    call_rows = []

    for K in sorted([x for x in calls.index if x > K0]):
        row = calls.loc[K].copy()
        row["strike"] = K
        row["QK"] = row["mid"]
        call_rows.append(row)

    call_otm_raw = pd.DataFrame(call_rows)
    call_otm = _select_otm_options_with_bid_rule(call_otm_raw, ascending=True)

    if K0 not in calls.index or K0 not in puts.index:
        raise ValueError(f"K0={K0} missing call or put quote")

    k0_row = calls.loc[K0].copy()
    k0_row["strike"] = K0
    k0_row["QK"] = (calls.loc[K0, "mid"] + puts.loc[K0, "mid"]) / 2
    k0_row["bid"] = (calls.loc[K0, "bid"] + puts.loc[K0, "bid"]) / 2
    k0_row["ask"] = (calls.loc[K0, "ask"] + puts.loc[K0, "ask"]) / 2
    k0_row["right"] = "K0_AVG"

    selected_options = pd.concat(
        [
            put_otm,
            pd.DataFrame([k0_row]),
            call_otm,
        ],
        ignore_index=True,
    )

    selected_options = selected_options.sort_values("strike").reset_index(drop=True)

    if len(selected_options) < 2:
        raise ValueError("Insufficient selected options for variance calculation")

    strikes = selected_options["strike"].astype(float).values
    QK = selected_options["QK"].astype(float).values

    delta_K = np.zeros(len(strikes))

    for i in range(len(strikes)):
        if i == 0:
            delta_K[i] = strikes[i + 1] - strikes[i]
        elif i == len(strikes) - 1:
            delta_K[i] = strikes[i] - strikes[i - 1]
        else:
            delta_K[i] = (strikes[i + 1] - strikes[i - 1]) / 2

    selected_options["delta_K"] = delta_K

    selected_options["contribution"] = (
        selected_options["delta_K"]
        / (selected_options["strike"] ** 2)
        * math.exp(r * T)
        * selected_options["QK"]
    )

    variance = (
        (2 / T) * selected_options["contribution"].sum()
        - (1 / T) * ((F / K0) - 1) ** 2
    )

    return {
        "variance": variance,
        "F": F,
        "K0": K0,
        "K_star": K_star,
        "T": T,
        "minutes": minutes_to_expiry,
        "num_options": len(selected_options),
        "selected_options": selected_options,
    }

In [42]:
# ============================================================
# Retest failing chain after robust forward patch
# ============================================================

calc = calc_single_term_variance(
    chain=debug_chain,
    minutes_to_expiry=debug_minutes,
    r=debug_rate,
)

print("Variance:", calc["variance"])
print("Forward:", calc["F"])
print("K0:", calc["K0"])
print("K_star:", calc["K_star"])
print("Num selected options:", calc["num_options"])

Variance: 0.08614338228909249
Forward: 4300.0
K0: 4300.0
K_star: 4300.0
Num selected options: 176


In [43]:
# ============================================================
# Retry 2022-02-22 after robust forward patch
# ============================================================

retry_date = 20220222

history_df, retry_errors_df = update_term_structure_history_for_range_v7(
    start_date=retry_date,
    end_date=retry_date,
    target_tenor_days=TARGET_TENOR_DAYS,
    calc_time_ms=CALC_TIME_MS,
    rate_symbol=DEFAULT_RATE_SYMBOL,
    max_workers=1,
    force_refresh=False,
    continue_on_error=True,
)

print("Rows in history:", len(history_df))
print("Retry errors:", len(retry_errors_df))
display(retry_errors_df)

Candidate dates: 1
Missing/incomplete v0.7 dates: 1
[1/1] Calculating 20220222...
Saved CSV: C:\Users\patri\vrp_project\data\processed\vix_term_structure_history.csv
Saved parquet: C:\Users\patri\vrp_project\data\processed\vix_term_structure_history.parquet
Rows in updated history: 10251
Min date: 20180625
Max date: 20221230
Rows in history: 10251
Retry errors: 0


""


In [44]:
# ============================================================
# Validate v0.7 market-close 2021-2022 backfill
# ============================================================

history_check_df = load_existing_term_structure_history()

start_date = 20210101
end_date = 20221231

trade_dates_range = get_spx_trade_dates_between(
    start_date=start_date,
    end_date=end_date
)

slice_range = history_check_df[
    (history_check_df["trade_date"] >= start_date)
    & (history_check_df["trade_date"] <= end_date)
    & (history_check_df["methodology_version"] == METHODOLOGY_VERSION)
].copy()

print("Rows:", len(slice_range))
print("Unique trade dates:", slice_range["trade_date"].nunique())
print("Expected trade dates:", len(trade_dates_range))
print("Expected rows:", len(trade_dates_range) * len(TARGET_TENOR_DAYS))

counts_range = (
    slice_range
    .groupby("trade_date")
    .agg(
        rows=("target_days", "count"),
        unique_tenors=("target_days", "nunique"),
        min_tenor=("target_days", "min"),
        max_tenor=("target_days", "max")
    )
    .reset_index()
)

bad_dates = counts_range[
    (counts_range["rows"] != len(TARGET_TENOR_DAYS))
    | (counts_range["unique_tenors"] != len(TARGET_TENOR_DAYS))
    | (counts_range["min_tenor"] != min(TARGET_TENOR_DAYS))
    | (counts_range["max_tenor"] != max(TARGET_TENOR_DAYS))
]

duplicate_check = (
    slice_range
    .groupby(["trade_date", "target_days", "methodology_version", "quote_time"])
    .size()
    .reset_index(name="count")
)

duplicates = duplicate_check[duplicate_check["count"] > 1]

print("Bad/incomplete dates:", len(bad_dates))
display(bad_dates)

print("Duplicate key rows:", len(duplicates))
display(duplicates)

print("Quote-time offsets in this range:")
display(slice_range["quote_time_offset_minutes"].value_counts().sort_index())

print("Market-close times in this range:")
display(slice_range["market_close_time"].value_counts())

Rows: 4527
Unique trade dates: 503
Expected trade dates: 503
Expected rows: 4527
Bad/incomplete dates: 0


,trade_date,rows,unique_tenors,min_tenor,max_tenor


Duplicate key rows: 0


,trade_date,target_days,methodology_version,quote_time,count


Quote-time offsets in this range:


quote_time_offset_minutes
0.0     9
1.0    18
Name: count, dtype: int64

Market-close times in this range:


market_close_time
16:00    4509
13:00      18
Name: count, dtype: int64

In [45]:
# ============================================================
# Fill missing quote-time audit metadata
# ============================================================

processed_history_csv = PROCESSED_DATA_DIR / "vix_term_structure_history.csv"
processed_history_parquet = PROCESSED_DATA_DIR / "vix_term_structure_history.parquet"

history_df = load_existing_term_structure_history().copy()

print("Rows before metadata fill:", len(history_df))

# Ensure columns exist
if "market_close_time" not in history_df.columns:
    history_df["market_close_time"] = pd.NA

if "quote_time_used" not in history_df.columns:
    history_df["quote_time_used"] = pd.NA

if "quote_time_offset_minutes" not in history_df.columns:
    history_df["quote_time_offset_minutes"] = pd.NA

if "calc_time_ms" not in history_df.columns:
    history_df["calc_time_ms"] = pd.NA

# Build close-time lookup for all trade dates
close_info = {
    int(date): get_market_close_time_for_trade_date(int(date))
    for date in history_df["trade_date"].drop_duplicates()
}

# Fill missing market close time / calc time
history_df["market_close_time"] = history_df.apply(
    lambda row: close_info[int(row["trade_date"])][0]
    if pd.isna(row["market_close_time"])
    else row["market_close_time"],
    axis=1
)

history_df["calc_time_ms"] = history_df.apply(
    lambda row: close_info[int(row["trade_date"])][1]
    if pd.isna(row["calc_time_ms"])
    else row["calc_time_ms"],
    axis=1
)

# For rows calculated before the fallback patch, assume quote time used = market close
# unless already populated otherwise.
history_df["quote_time_used"] = history_df.apply(
    lambda row: row["market_close_time"]
    if pd.isna(row["quote_time_used"])
    else row["quote_time_used"],
    axis=1
)

history_df["quote_time_offset_minutes"] = history_df["quote_time_offset_minutes"].fillna(0)

# Preserve current methodology labels
history_df["methodology_version"] = METHODOLOGY_VERSION
history_df["quote_time"] = QUOTE_TIME_LABEL

history_df.to_csv(processed_history_csv, index=False)
history_df.to_parquet(processed_history_parquet, index=False)

history_df = load_existing_term_structure_history()

print("Rows after metadata fill:", len(history_df))

print("Quote-time offsets:")
display(history_df["quote_time_offset_minutes"].value_counts(dropna=False).sort_index())

print("Quote times used:")
display(history_df["quote_time_used"].value_counts(dropna=False).sort_index())

print("Market-close times:")
display(history_df["market_close_time"].value_counts(dropna=False).sort_index())

Rows before metadata fill: 10251
Rows after metadata fill: 10251
Quote-time offsets:


quote_time_offset_minutes
0.0    10233
1.0       18
Name: count, dtype: int64

Quote times used:


quote_time_used
13:00       90
15:59       18
16:00    10143
Name: count, dtype: int64

Market-close times:


market_close_time
13:00       90
16:00    10161
Name: count, dtype: int64

In [46]:
# ============================================================
# v0.7 market-close backfill: 2023-2024
# ============================================================

BACKFILL_START_DATE = 20230101
BACKFILL_END_DATE = 20241231

history_df, errors_df = update_term_structure_history_for_range_v7(
    start_date=BACKFILL_START_DATE,
    end_date=BACKFILL_END_DATE,
    target_tenor_days=TARGET_TENOR_DAYS,
    calc_time_ms=CALC_TIME_MS,
    rate_symbol=DEFAULT_RATE_SYMBOL,
    max_workers=4,
    force_refresh=False,
    continue_on_error=True,
)

print("Rows in history:", len(history_df))
print("Errors:", len(errors_df))
display(errors_df)

Candidate dates: 502
Missing/incomplete v0.7 dates: 502
[1/502] Calculating 20230103...
[2/502] Calculating 20230104...
[3/502] Calculating 20230105...
[4/502] Calculating 20230106...
[5/502] Calculating 20230109...
[6/502] Calculating 20230110...
[7/502] Calculating 20230111...
[8/502] Calculating 20230112...
[9/502] Calculating 20230113...
[10/502] Calculating 20230117...
[11/502] Calculating 20230118...
[12/502] Calculating 20230119...
[13/502] Calculating 20230120...
[14/502] Calculating 20230123...
[15/502] Calculating 20230124...
[16/502] Calculating 20230125...
[17/502] Calculating 20230126...
[18/502] Calculating 20230127...
[19/502] Calculating 20230130...
[20/502] Calculating 20230131...
[21/502] Calculating 20230201...
[22/502] Calculating 20230202...
[23/502] Calculating 20230203...
[24/502] Calculating 20230206...
[25/502] Calculating 20230207...
[26/502] Calculating 20230208...
[27/502] Calculating 20230209...
[28/502] Calculating 20230210...
[29/502] Calculating 20230213

""


In [47]:
# ============================================================
# Validate v0.7 market-close 2023-2024 backfill
# ============================================================

history_check_df = load_existing_term_structure_history()

start_date = 20230101
end_date = 20241231

trade_dates_range = get_spx_trade_dates_between(
    start_date=start_date,
    end_date=end_date
)

slice_range = history_check_df[
    (history_check_df["trade_date"] >= start_date)
    & (history_check_df["trade_date"] <= end_date)
    & (history_check_df["methodology_version"] == METHODOLOGY_VERSION)
].copy()

print("Rows:", len(slice_range))
print("Unique trade dates:", slice_range["trade_date"].nunique())
print("Expected trade dates:", len(trade_dates_range))
print("Expected rows:", len(trade_dates_range) * len(TARGET_TENOR_DAYS))

counts_range = (
    slice_range
    .groupby("trade_date")
    .agg(
        rows=("target_days", "count"),
        unique_tenors=("target_days", "nunique"),
        min_tenor=("target_days", "min"),
        max_tenor=("target_days", "max")
    )
    .reset_index()
)

bad_dates = counts_range[
    (counts_range["rows"] != len(TARGET_TENOR_DAYS))
    | (counts_range["unique_tenors"] != len(TARGET_TENOR_DAYS))
    | (counts_range["min_tenor"] != min(TARGET_TENOR_DAYS))
    | (counts_range["max_tenor"] != max(TARGET_TENOR_DAYS))
]

duplicate_check = (
    slice_range
    .groupby(["trade_date", "target_days", "methodology_version", "quote_time"])
    .size()
    .reset_index(name="count")
)

duplicates = duplicate_check[duplicate_check["count"] > 1]

print("Bad/incomplete dates:", len(bad_dates))
display(bad_dates)

print("Duplicate key rows:", len(duplicates))
display(duplicates)

print("Quote-time offsets in this range:")
display(slice_range["quote_time_offset_minutes"].value_counts().sort_index())

print("Market-close times in this range:")
display(slice_range["market_close_time"].value_counts())


Rows: 4518
Unique trade dates: 502
Expected trade dates: 502
Expected rows: 4518
Bad/incomplete dates: 0


,trade_date,rows,unique_tenors,min_tenor,max_tenor


Duplicate key rows: 0


,trade_date,target_days,methodology_version,quote_time,count


Quote-time offsets in this range:


quote_time_offset_minutes
0.0    4500
1.0       9
2.0       9
Name: count, dtype: int64

Market-close times in this range:


market_close_time
16:00    4473
13:00      45
Name: count, dtype: int64

In [48]:
# ============================================================
# v0.7 market-close backfill: 2025 through 2026-06-25
# ============================================================

BACKFILL_START_DATE = 20250101
BACKFILL_END_DATE = 20260625

history_df, errors_df = update_term_structure_history_for_range_v7(
    start_date=BACKFILL_START_DATE,
    end_date=BACKFILL_END_DATE,
    target_tenor_days=TARGET_TENOR_DAYS,
    calc_time_ms=CALC_TIME_MS,
    rate_symbol=DEFAULT_RATE_SYMBOL,
    max_workers=4,
    force_refresh=False,
    continue_on_error=True,
)

print("Rows in history:", len(history_df))
print("Errors:", len(errors_df))
display(errors_df)

Candidate dates: 370
Missing/incomplete v0.7 dates: 370
[1/370] Calculating 20250102...
[2/370] Calculating 20250103...
[3/370] Calculating 20250106...
[4/370] Calculating 20250107...
[5/370] Calculating 20250108...
[6/370] Calculating 20250110...
[7/370] Calculating 20250113...
[8/370] Calculating 20250114...
[9/370] Calculating 20250115...
[10/370] Calculating 20250116...
[11/370] Calculating 20250117...
[12/370] Calculating 20250121...
[13/370] Calculating 20250122...
[14/370] Calculating 20250123...
[15/370] Calculating 20250124...
[16/370] Calculating 20250127...
[17/370] Calculating 20250128...
[18/370] Calculating 20250129...
[19/370] Calculating 20250130...
[20/370] Calculating 20250131...
[21/370] Calculating 20250203...
[22/370] Calculating 20250204...
[23/370] Calculating 20250205...
[24/370] Calculating 20250206...
[25/370] Calculating 20250207...
[26/370] Calculating 20250210...
[27/370] Calculating 20250211...
[28/370] Calculating 20250212...
[29/370] Calculating 20250213

,trade_date,error
0,20260223,ThetaData option-chain request failed.\nURL: h...


In [49]:
# ============================================================
# Retry final failed date: 2026-02-23
# ============================================================

retry_date = 20260223

history_df, retry_errors_df = update_term_structure_history_for_range_v7(
    start_date=retry_date,
    end_date=retry_date,
    target_tenor_days=TARGET_TENOR_DAYS,
    calc_time_ms=CALC_TIME_MS,
    rate_symbol=DEFAULT_RATE_SYMBOL,
    max_workers=1,
    force_refresh=True,
    continue_on_error=True,
)

print("Rows in history:", len(history_df))
print("Retry errors:", len(retry_errors_df))
display(retry_errors_df)

Candidate dates: 1
Missing/incomplete v0.7 dates: 1
[1/1] Calculating 20260223...
ERROR on 20260223: ThetaData option-chain request failed.
URL: http://127.0.0.1:25503/v3/option/history/quote?symbol=SPXW&expiration=2026-04-02&strike=%2A&right=both&start_date=2026-02-23&end_date=2026-02-23&start_time=15%3A50%3A00.000&end_time=15%3A50%3A00.000&interval=1m&format=json
Status code: 472
Response text:
No data found for your request
Rows in history: 18090
Retry errors: 1


,trade_date,error
0,20260223,ThetaData option-chain request failed.\nURL: h...


In [50]:
# ============================================================
# v0.7 unavailable-chain exclusion patch
# Handles listed calendar expirations that have no ThetaData chain
# ============================================================

# Keep original candidate function once
try:
    _get_friday_cycle_expiration_candidates_unfiltered
except NameError:
    _get_friday_cycle_expiration_candidates_unfiltered = get_friday_cycle_expiration_candidates


UNAVAILABLE_CHAINS = {
    # trade_date, root, expiration
    (20260223, "SPXW", 20260402),
}


def get_friday_cycle_expiration_candidates(
    trade_date,
    calc_time_ms=CALC_TIME_MS,
    min_minutes_to_expiry=1,
    spx_trading_dates=None,
):
    """
    Friday-cycle expiration candidates with known unavailable vendor chains removed.

    This keeps the methodology intact but avoids selecting an expiration for which
    ThetaData returns "No data found" on the specific trade date.
    """
    candidates = _get_friday_cycle_expiration_candidates_unfiltered(
        trade_date=trade_date,
        calc_time_ms=calc_time_ms,
        min_minutes_to_expiry=min_minutes_to_expiry,
        spx_trading_dates=spx_trading_dates,
    ).copy()

    if len(candidates) == 0:
        return candidates

    mask_unavailable = candidates.apply(
        lambda row: (
            int(trade_date),
            str(row["root"]),
            int(row["expiration"]),
        ) in UNAVAILABLE_CHAINS,
        axis=1,
    )

    removed = candidates[mask_unavailable].copy()

    if len(removed) > 0:
        print("Excluding unavailable ThetaData chains:")
        display(removed)

    candidates = candidates[~mask_unavailable].copy()

    if candidates.empty:
        raise ValueError(
            f"No Friday-cycle expirations remain after unavailable-chain filter for {trade_date}"
        )

    return candidates.sort_values("minutes").reset_index(drop=True)

In [51]:
# ============================================================
# Inspect adjusted candidates for 2026-02-23
# ============================================================

adjusted_candidates = get_friday_cycle_expiration_candidates(
    trade_date=20260223,
    calc_time_ms=get_market_close_time_for_trade_date(20260223)[1],
)

display(adjusted_candidates.head(12))

required_by_tenor, unique_chains = get_required_chains_for_target_tenors(
    trade_date=20260223,
    target_tenor_days=TARGET_TENOR_DAYS,
    calc_time_ms=get_market_close_time_for_trade_date(20260223)[1],
)

display(required_by_tenor)
display(unique_chains)

Excluding unavailable ThetaData chains:


,root,expiration,minutes,days
5,SPXW,20260402,54720,38.0


,root,expiration,minutes,days
0,SPXW,20260227,5760,4.000000
1,SPXW,20260306,15840,11.000000
2,SPXW,20260313,25920,18.000000
3,SPX,20260320,35610,24.729167
4,SPXW,20260327,46080,32.000000
5,SPXW,20260410,66240,46.000000
6,SPX,20260417,75930,52.729167
7,SPXW,20260424,86400,60.000000
8,SPXW,20260501,96480,67.000000
9,SPXW,20260508,106560,74.000000


Excluding unavailable ThetaData chains:


,root,expiration,minutes,days
5,SPXW,20260402,54720,38.0


...

,root,expiration,minutes,days
5,SPXW,20260402,54720,38.0


Excluding unavailable ThetaData chains:


,root,expiration,minutes,days
5,SPXW,20260402,54720,38.0


,target_days,leg,root,expiration,minutes,days
0,9,near,SPXW,20260227,5760,4.000000
1,9,next,SPXW,20260306,15840,11.000000
2,12,near,SPXW,20260306,15840,11.000000
3,12,next,SPXW,20260313,25920,18.000000
4,15,near,SPXW,20260306,15840,11.000000
5,15,next,SPXW,20260313,25920,18.000000
6,18,near,SPXW,20260313,25920,18.000000
7,18,next,SPX,20260320,35610,24.729167
8,21,near,SPXW,20260313,25920,18.000000
9,21,next,SPX,20260320,35610,24.729167


,root,expiration,minutes,days
0,SPXW,20260227,5760,4.000000
1,SPXW,20260306,15840,11.000000
2,SPXW,20260313,25920,18.000000
3,SPX,20260320,35610,24.729167
4,SPXW,20260327,46080,32.000000
5,SPXW,20260410,66240,46.000000


In [52]:
# ============================================================
# Retry 2026-02-23 after excluding unavailable chain
# ============================================================

retry_date = 20260223

history_df, retry_errors_df = update_term_structure_history_for_range_v7(
    start_date=retry_date,
    end_date=retry_date,
    target_tenor_days=TARGET_TENOR_DAYS,
    calc_time_ms=CALC_TIME_MS,
    rate_symbol=DEFAULT_RATE_SYMBOL,
    max_workers=1,
    force_refresh=False,
    continue_on_error=True,
)

print("Rows in history:", len(history_df))
print("Retry errors:", len(retry_errors_df))
display(retry_errors_df)

Candidate dates: 1
Missing/incomplete v0.7 dates: 1
[1/1] Calculating 20260223...
Excluding unavailable ThetaData chains:


,root,expiration,minutes,days
5,SPXW,20260402,54720,38.0


Excluding unavailable ThetaData chains:


,root,expiration,minutes,days
5,SPXW,20260402,54720,38.0


Excluding unavailable ThetaData chains:


...

,root,expiration,minutes,days
5,SPXW,20260402,54720,38.0


Excluding unavailable ThetaData chains:


,root,expiration,minutes,days
5,SPXW,20260402,54720,38.0


Saved CSV: C:\Users\patri\vrp_project\data\processed\vix_term_structure_history.csv
Saved parquet: C:\Users\patri\vrp_project\data\processed\vix_term_structure_history.parquet
Rows in updated history: 18099
Min date: 20180625
Max date: 20260625
Rows in history: 18099
Retry errors: 0


""


In [53]:
# ============================================================
# Validate v0.7 market-close 2025 through 2026-06-25 backfill
# ============================================================

history_check_df = load_existing_term_structure_history()

start_date = 20250101
end_date = 20260625

trade_dates_range = get_spx_trade_dates_between(
    start_date=start_date,
    end_date=end_date
)

slice_range = history_check_df[
    (history_check_df["trade_date"] >= start_date)
    & (history_check_df["trade_date"] <= end_date)
    & (history_check_df["methodology_version"] == METHODOLOGY_VERSION)
].copy()

print("Rows:", len(slice_range))
print("Unique trade dates:", slice_range["trade_date"].nunique())
print("Expected trade dates:", len(trade_dates_range))
print("Expected rows:", len(trade_dates_range) * len(TARGET_TENOR_DAYS))

counts_range = (
    slice_range
    .groupby("trade_date")
    .agg(
        rows=("target_days", "count"),
        unique_tenors=("target_days", "nunique"),
        min_tenor=("target_days", "min"),
        max_tenor=("target_days", "max")
    )
    .reset_index()
)

bad_dates = counts_range[
    (counts_range["rows"] != len(TARGET_TENOR_DAYS))
    | (counts_range["unique_tenors"] != len(TARGET_TENOR_DAYS))
    | (counts_range["min_tenor"] != min(TARGET_TENOR_DAYS))
    | (counts_range["max_tenor"] != max(TARGET_TENOR_DAYS))
]

duplicate_check = (
    slice_range
    .groupby(["trade_date", "target_days", "methodology_version", "quote_time"])
    .size()
    .reset_index(name="count")
)

duplicates = duplicate_check[duplicate_check["count"] > 1]

print("Bad/incomplete dates:", len(bad_dates))
display(bad_dates)

print("Duplicate key rows:", len(duplicates))
display(duplicates)

print("Quote-time offsets in this range:")
display(slice_range["quote_time_offset_minutes"].value_counts().sort_index())

print("Market-close times in this range:")
display(slice_range["market_close_time"].value_counts())

Rows: 3330
Unique trade dates: 370
Expected trade dates: 370
Expected rows: 3330
Bad/incomplete dates: 0


,trade_date,rows,unique_tenors,min_tenor,max_tenor


Duplicate key rows: 0


,trade_date,target_days,methodology_version,quote_time,count


Quote-time offsets in this range:


quote_time_offset_minutes
0.0    3330
Name: count, dtype: int64

Market-close times in this range:


market_close_time
16:00    3303
13:00      27
Name: count, dtype: int64

In [54]:
# ============================================================
# Validate full v0.7 market-close history
# ============================================================

history_check_df = load_existing_term_structure_history()

start_date = 20180625
end_date = 20260625

all_trade_dates = get_spx_trade_dates_between(
    start_date=start_date,
    end_date=end_date
)

full_slice = history_check_df[
    (history_check_df["trade_date"] >= start_date)
    & (history_check_df["trade_date"] <= end_date)
    & (history_check_df["methodology_version"] == METHODOLOGY_VERSION)
].copy()

print("Full rows:", len(full_slice))
print("Full unique trade dates:", full_slice["trade_date"].nunique())
print("Expected trade dates:", len(all_trade_dates))
print("Expected rows:", len(all_trade_dates) * len(TARGET_TENOR_DAYS))

full_counts = (
    full_slice
    .groupby("trade_date")
    .agg(
        rows=("target_days", "count"),
        unique_tenors=("target_days", "nunique"),
        min_tenor=("target_days", "min"),
        max_tenor=("target_days", "max")
    )
    .reset_index()
)

bad_full_dates = full_counts[
    (full_counts["rows"] != len(TARGET_TENOR_DAYS))
    | (full_counts["unique_tenors"] != len(TARGET_TENOR_DAYS))
    | (full_counts["min_tenor"] != min(TARGET_TENOR_DAYS))
    | (full_counts["max_tenor"] != max(TARGET_TENOR_DAYS))
]

full_duplicate_check = (
    full_slice
    .groupby(["trade_date", "target_days", "methodology_version", "quote_time"])
    .size()
    .reset_index(name="count")
)

full_duplicates = full_duplicate_check[full_duplicate_check["count"] > 1]

print("Bad/incomplete full-history dates:", len(bad_full_dates))
display(bad_full_dates)

print("Duplicate key rows:", len(full_duplicates))
display(full_duplicates)

print("Quote-time offsets full history:")
display(full_slice["quote_time_offset_minutes"].value_counts().sort_index())

print("Market-close times full history:")
display(full_slice["market_close_time"].value_counts())

print("Methodology versions:")
display(full_slice["methodology_version"].value_counts())

print("Rate sources:")
display(full_slice["rate_source"].value_counts())

print("Option sources:")
display(full_slice["option_source"].value_counts())

Full rows: 18099
Full unique trade dates: 2011
Expected trade dates: 2011
Expected rows: 18099
Bad/incomplete full-history dates: 0


,trade_date,rows,unique_tenors,min_tenor,max_tenor


Duplicate key rows: 0


,trade_date,target_days,methodology_version,quote_time,count


Quote-time offsets full history:


...

methodology_version
v0.7_exchange_calendar_fred_sofr_market_close    18099
Name: count, dtype: int64

Rate sources:


rate_source
FRED_SOFR    18099
Name: count, dtype: int64

Option sources:


option_source
THETADATA_SPX_SPXW    18099
Name: count, dtype: int64